# RT-RPINN: bending-dominated inverse identification\nSelf-contained seed-42 notebook matching the final EX5 runner: shift-only inverse, fixed C2 and Prony spectrum, reaction-moment weight 1.\n\nThis notebook is organized into modular blocks following the EX2/EX4 style. The code is unchanged; only the notebook structure is separated for readability.\n

## Import Required Libraries and Problem Setup\n\nSet notebook paths, describe the EX5 bending problem, import libraries, and initialize the compute device.\n

In [ ]:
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()

"""
Example 5: Inverse Identification of Time-Temperature Coupling Parameters
from a BENDING shape-memory cycle, using a Spatiotemporal PINN.

=============================================================================
PROBLEM DESCRIPTION
=============================================================================

This is a single-seed, non-LSTM, total-Lagrangian inverse problem. The inverse
gradient follows the network displacement field through finite-strain
kinematics, reduced-time recursion, first-Piola stress, reference equilibrium,
and reference-boundary tractions. FE strain and FE stress are not training data.

Specimen: 95×13×2 mm, 45° off-axis SMPC (long slender beam)
Loading:  Full shape-memory cycle driven by end rotation at RP2
  Step 1 (0–20 s):   Programming at 343 K, ramp end rotation UR to 4.71 rad
  Step 2 (20–70 s):  Constrained cooling 343→298 K (curvature held)
  Step 3 (70–71 s):  Unloading at 298 K (elastic spring-back, 1 s)
  Step 4 (71–121 s): Free thermal recovery 298→353 K

OBSERVABLES (EX5 differs from EX4):
  - Boundary observable is the REACTION MOMENT (RM) at RP2, NOT a reaction force.
  - End-kinematics observable is the ROTATION (UR) at RP2, NOT a displacement.
  - Shift parameters and free Prony amplitudes are identified from full-field
    displacement observations, total-Lagrangian mechanics, and the EX5 reaction-
    moment magnitude at RP2. The moment is integrated as r x (P N) over the
    reference end face; mean(stress)*area is not used.

TRUE SHIFT MODEL (from UMAT Source-C6-202301.for):
  T > 317.4 K  →  WLF:       log10(aT) = -C1*(T - Tref) / (C2 + (T - Tref))
  T ≤ 317.4 K  →  Arrhenius: aT = exp(Ea_R * (1/T - 1/Tref_arr))
  True C1    = 14.8
  True C2    = 45.6 K
  True Ea_R  = 27403.3 K
  True Tref  = 323 K  (WLF reference)
  True Tref_arr = 336 K (Arrhenius reference)
  T_transition  = 317.4 K (crossover)

TRAINABLE SHIFT PARAMETERS (C2 is fixed):
  log_C1:    log of WLF C1  (init: log(8.0),     true: log(14.8))
  log_C2:    fixed at log(45.6)
  log_Ea_R:  log of Ea/R    (init: log(20000.0), true: log(27403.3))

MATERIAL PARAMETERS:
  All elastic constants: C11=11250.16, C12=891.26, C22=1425.85, C23=915.56, C66=267.69 MPa
  Prony times are fixed. g0, g1, and g5 are fixed; g2, g3, g4, and g∞
  are trainable under positivity and sum-to-one constraints.

KEY DIFFERENCE FROM EX2:
  - EX2 is isothermal → a_T = 1, q recursion uses real time
  - EX5 is non-isothermal → a_T(T), q recursion uses REDUCED time dξ = dt/a_T
  - Network input extended to (x,y,z,t,T): temperature drives shift factor
  - Primary inverse signal: temperature-dependent recovery of the full field

=============================================================================
"""

import sys
import math
import time
import re
import argparse
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path


# ---------------------------------------------------------------------------


## Logging and Reproducibility Utilities\n\nDefine stdout logging and random-seed utilities used by the benchmark.\n

In [ ]:
# Tee: mirror stdout to log file
# ---------------------------------------------------------------------------
class _Tee:
    def __init__(self, original, file_path):
        self._orig = original
        self._file = open(file_path, 'w', buffering=1, encoding='utf-8')

    def write(self, data):
        self._orig.write(data)
        self._file.write(data)

    def flush(self):
        self._orig.flush()
        self._file.flush()

    def close(self):
        self._file.close()

    def __getattr__(self, name):
        return getattr(self._orig, name)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------------


## FEDataLoader\n\nLoad the multi-step bending-cycle FE data, reaction moment/end-rotation histories, and sparse spatial/temporal samples.\n

In [ ]:
# 1. FE Data Loader — multi-step non-isothermal cycle
# ---------------------------------------------------------------------------

# Temperature protocol derived from Abaqus step definitions
_STEP_T_FN = {
    1: lambda t: 343.0,
    2: lambda t: 343.0 - (343.0 - 298.0) * (t - 20.0) / 50.0,
    3: lambda t: 298.0,
    4: lambda t: 298.0 + (353.0 - 298.0) * (t - 71.0) / 50.0,
}
# Step time boundaries (EX5: durations 20 / 50 / 1 / 50 s → total 121 s)
_STEP_BOUNDS = {1: (0.0, 20.0), 2: (20.0, 70.0), 3: (70.0, 71.0), 4: (71.0, 121.0)}
# Time during Step-2 cooling when T crosses T_cross=317.4 K (WLF→Arrhenius boundary)
# t = 20 + (343-317.4)/(343-298)*50 = 48.44 s
_T_CROSS_TIME = 48.44
# Last FE frame time (end of free recovery) used for transition-window weighting.
_T_END = 120.96
_STEP_PATTERNS = {
    1: '*_Step-1-High-temperature_frame*.csv',
    2: '*_Step-2-Cooling_frame*.csv',
    3: '*_Step-3-Low-temperature unloading_frame*.csv',
    4: '*_Step-4-Free recovery_frame*.csv',
}


class FEDataLoader:
    """Load and process FE simulation results from multi-step non-isothermal cycle."""

    def __init__(self, data_dir, rf_file, u_file, frame_time_file, stride=5,
                 n_spatial=None, spatial_seed=0, n_temporal=None):
        """
        Args:
            stride: Load every stride-th frame per step (default 5).
                    Frame 0 and the last frame of each step are always kept
                    to preserve step transitions.
                    stride=1 → all 1296 frames (slow, ~1296 CSVs read)
                    stride=5 → ~260 frames  (recommended for training)
                    stride=10 → ~130 frames (faster, coarser)
            n_spatial: Sparse SPATIAL subsampling — keep only this many nodes
                    (a single fixed node set across all frames). The full mesh has
                    14454 nodes; for this large-deformation bending case computing
                    over all of them every epoch is wasteful. The left and right end
                    faces (clamped / prescribed-rotation BC) are ALWAYS kept in full;
                    the remaining budget is filled with a random reference-config
                    sample. None → keep all nodes. Node identity is consistent across
                    frames (selection is by NodeLabel on the constant reference X,Y,Z).
            spatial_seed: RNG seed for the spatial subsample (reproducible).
            n_temporal: Sparse TEMPORAL subsampling — target total number of frames
                    to load across the whole 4-step cycle (~1296 available). When set,
                    it overrides `stride` with an effective stride = round(total /
                    n_temporal); each step's first/last frame is always kept to
                    preserve step transitions. None → use `stride`.
        """
        self.data_dir = Path(data_dir)
        self.rf_file = Path(rf_file)
        self.u_file = Path(u_file)
        self.frame_time_file = Path(frame_time_file)
        self.stride = max(1, int(stride))
        self.n_temporal = (int(n_temporal) if n_temporal else None)
        self.n_spatial = (int(n_spatial) if n_spatial else None)
        self.spatial_seed = int(spatial_seed)
        self.master_node_ids = None    # fixed sparse node set (None → keep all)
        self._node_id_col = None
        self.data = []
        self.step_frame_time = {}  # (step, local_frame) → time
        self.all_frame_times = None  # Full FE timeline from frame-time.csv

    # ------------------------------------------------------------------
    # Sparse spatial subsampling helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _detect_node_col(df):
        for col in ['NodeLabel', 'Node', 'NID']:
            if col in df.columns:
                return col
        return None

    def _build_master_nodes(self, df):
        """Select a fixed sparse node set from a reference-config frame.

        Always keeps the left (x=x_min) and right (x=x_max) end faces in full;
        fills the rest of the budget with a random sample. Sets self.master_node_ids
        (sorted np.array) or leaves it None if no subsampling is requested/possible.
        """
        self._node_id_col = self._detect_node_col(df)
        if self.n_spatial is None or self._node_id_col is None:
            return
        ids = df[self._node_id_col].values
        if self.n_spatial >= len(ids):
            print(f"  [Spatial] n_spatial={self.n_spatial} >= total nodes "
                  f"{len(ids)} → keeping all nodes.")
            return
        x = df['X'].values
        tol = 1e-4
        face_mask = (np.abs(x - x.min()) < tol) | (np.abs(x - x.max()) < tol)
        forced = ids[face_mask]
        remaining = ids[~np.isin(ids, forced)]
        n_extra = self.n_spatial - len(forced)
        rng = np.random.default_rng(self.spatial_seed)
        if n_extra <= 0:
            # Budget smaller than the end faces themselves: keep faces only.
            sel = np.unique(forced)
        elif n_extra >= len(remaining):
            sel = ids
        else:
            chosen = rng.choice(remaining, size=n_extra, replace=False)
            sel = np.concatenate([forced, chosen])
        self.master_node_ids = np.unique(sel)
        print(f"  [Spatial] sparse sampling: {len(self.master_node_ids)} / {len(ids)} nodes "
              f"(end faces kept in full: {len(np.unique(forced))}; seed={self.spatial_seed})")

    def load_all_data(self):
        """Load (strided) CSV files from the 4-step results directory."""
        self.step_load_stats = {}

        # ------------------------------------------------------------------
        # Build (step, local_frame) → time mapping
        # frame-time.csv contains rows like "local_frame, time" for each step
        # concatenated; step boundaries detected by local_frame resetting to 1.
        # ------------------------------------------------------------------
        raw_ft = pd.read_csv(self.frame_time_file, header=None,
                              names=['LocalFrame', 'Time'])
        step = 1
        prev_frame = 0
        step_ids = []
        for _, row in raw_ft.iterrows():
            lf = int(row['LocalFrame'])
            if lf < prev_frame:
                step += 1
            step_ids.append(step)
            prev_frame = lf
        raw_ft['Step'] = step_ids
        self.step_frame_time = {
            (int(r['Step']), int(r['LocalFrame'])): float(r['Time'])
            for _, r in raw_ft.iterrows()
        }
        # Add frame-0 entries (initial conditions, not in frame-time.csv)
        step_start_t = {1: 0.0, 2: 20.0, 3: 70.0, 4: 71.0}
        for s, t0 in step_start_t.items():
            self.step_frame_time[(s, 0)] = t0
        self.all_frame_times = np.array(
            sorted(set(float(t) for t in self.step_frame_time.values())),
            dtype=float
        )

        # ------------------------------------------------------------------
        # Load per-frame CSVs for each step
        # ------------------------------------------------------------------
        def frame_from_filename(fp):
            m = re.search(r'frame(\d+)', Path(fp).stem)
            if not m:
                raise ValueError(f"Cannot parse frame index from: {fp}")
            return int(m.group(1))

        # Pre-glob every step so we can size an effective temporal stride.
        step_files_map = {
            step_num: sorted(list(self.data_dir.glob(pattern)),
                             key=lambda f: frame_from_filename(f))
            for step_num, pattern in _STEP_PATTERNS.items()
        }
        total_available = sum(len(v) for v in step_files_map.values())
        if self.n_temporal is not None and total_available > 0:
            eff_stride = max(1, round(total_available / self.n_temporal))
            print(f"  [Temporal] sparse sampling: target ~{self.n_temporal} frames "
                  f"of {total_available} → effective stride={eff_stride} "
                  f"(per-step first/last always kept).")
        else:
            eff_stride = self.stride

        for step_num, step_files in step_files_map.items():
            if not step_files:
                continue

            self.step_load_stats[step_num] = {
                'n_loaded_frames': 0,
                't_min': float('inf'),
                't_max': float('-inf'),
                'T_min': float('inf'),
                'T_max': float('-inf'),
                'n_out_of_bound_time': 0,
            }

            # Always keep first and last frame; apply effective stride to the rest
            last_idx = len(step_files) - 1
            keep = set()
            keep.add(0)
            keep.add(last_idx)
            keep.update(range(0, last_idx + 1, eff_stride))
            keep_sorted = sorted(keep)

            for file_idx in keep_sorted:
                csv_file = step_files[file_idx]
                local_frame = frame_from_filename(csv_file)
                time_val = self.step_frame_time.get((step_num, local_frame), None)
                if time_val is None:
                    continue  # No time mapping → skip

                df = pd.read_csv(csv_file)
                # Sparse spatial subsampling: build the fixed node set from the
                # first (reference-config) frame, then filter every frame to it.
                if self.n_spatial is not None and self.master_node_ids is None:
                    self._build_master_nodes(df)
                if self.master_node_ids is not None:
                    df = df[df[self._node_id_col].isin(self.master_node_ids)]
                df['Time'] = time_val
                df['Frame'] = local_frame
                df['Step'] = step_num
                T_fn = _STEP_T_FN[step_num]
                T_val = float(T_fn(time_val))
                if step_num == 2:
                    T_val = min(343.0, max(298.0, T_val))
                elif step_num == 4:
                    T_val = min(353.0, max(298.0, T_val))
                df['Temperature'] = T_val
                self.data.append(df)

                # Step-level load sanity stats
                st = self.step_load_stats[step_num]
                st['n_loaded_frames'] += 1
                st['t_min'] = min(st['t_min'], float(time_val))
                st['t_max'] = max(st['t_max'], float(time_val))
                st['T_min'] = min(st['T_min'], float(T_val))
                st['T_max'] = max(st['T_max'], float(T_val))
                t_lo, t_hi = _STEP_BOUNDS[step_num]
                if (time_val < t_lo - 1e-3) or (time_val > t_hi + 1e-3):
                    st['n_out_of_bound_time'] += 1

        if not self.data:
            raise RuntimeError(f"No data loaded from {self.data_dir}. "
                               "Check step CSV filenames match expected patterns.")

        # Sort by time to ensure chronological order
        self.data.sort(key=lambda df: df['Time'].iloc[0])

        self.full_data = pd.concat(self.data, ignore_index=True)

        # ------------------------------------------------------------------
        # Load RM (reaction moment) and UR (end rotation) history.
        # NOTE: EX5 observables are RM/UR, not RF/U. Column labels 'RF'/'U' are
        # kept as internal keys only; they hold RM [N·mm] and UR [rad] values
        # and are used for reaction-moment supervision / validation.
        # ------------------------------------------------------------------
        rf_df = pd.read_csv(self.rf_file, header=None, names=['Time', 'RF'])
        u_df = pd.read_csv(self.u_file, header=None, names=['Time', 'U'])

        # Match RM/UR times to the COMPLETE FE timeline, not the strided field subset.
        # Otherwise stride>1 creates artificial time misalignment warnings.
        all_times = self.all_frame_times

        def match_to_times(query_times, tol=1e-4):
            query_times = np.asarray(query_times, dtype=float)
            idx = np.searchsorted(all_times, query_times)
            idx = np.clip(idx, 1, len(all_times) - 1)
            left = idx - 1
            choose_right = (np.abs(all_times[idx] - query_times) <
                            np.abs(all_times[left] - query_times))
            best = np.where(choose_right, idx, left)
            matched = all_times[best]
            err = np.abs(matched - query_times)
            if np.any(err > tol):
                n_bad = int(np.sum(err > tol))
                print(f"  [Warning] {n_bad} RF/U times exceed tol={tol:.2e}. "
                      f"Max err={err.max():.3e}")
            return matched

        rf_df['MatchedTime'] = match_to_times(rf_df['Time'].values)
        u_df['MatchedTime'] = match_to_times(u_df['Time'].values)
        self.rf_data = rf_df.rename(columns={'MatchedTime': 'AlignedTime'})
        self.u_data = u_df.rename(columns={'MatchedTime': 'AlignedTime'})

        print(f"\nFE data loaded (stride={self.stride}):")
        print(f"  Total frames : {len(self.data)}")
        print(f"  Total nodes  : {len(self.full_data['NodeLabel'].unique())} unique")
        print(f"  Time range   : [{self.full_data['Time'].min():.2f}, "
              f"{self.full_data['Time'].max():.2f}] s")
        print(f"  Temp range   : [{self.full_data['Temperature'].min():.1f}, "
              f"{self.full_data['Temperature'].max():.1f}] K")
        print(f"  RM range     : [{rf_df['RF'].min():.3f}, {rf_df['RF'].max():.3f}] N·mm")
        print("  Step summary :")
        for s in sorted(self.step_load_stats.keys()):
            st = self.step_load_stats[s]
            print(
                f"    Step{s}: frames={st['n_loaded_frames']} "
                f"t=[{st['t_min']:.2f},{st['t_max']:.2f}] "
                f"T=[{st['T_min']:.1f},{st['T_max']:.1f}] "
                f"out_of_bound_time={st['n_out_of_bound_time']}"
            )

        return self.full_data

    def get_domain_bounds(self):
        return {
            'x_min': self.full_data['X'].min(),
            'x_max': self.full_data['X'].max(),
            'y_min': self.full_data['Y'].min(),
            'y_max': self.full_data['Y'].max(),
            'z_min': self.full_data['Z'].min(),
            'z_max': self.full_data['Z'].max(),
            't_min': self.full_data['Time'].min(),
            't_max': self.full_data['Time'].max(),
            'T_min': self.full_data['Temperature'].min(),
            'T_max': self.full_data['Temperature'].max(),
        }

    def get_rf_by_time(self, time_val, tol=1e-4):
        """Return RM (reaction moment, N·mm) value closest to time_val."""
        diff = np.abs(self.rf_data['AlignedTime'].values - time_val)
        idx = int(np.argmin(diff))
        return float(self.rf_data['RF'].iloc[idx])

    def get_u_by_time(self, time_val, tol=1e-4):
        """Return loaded-end rotation UR (rad) closest to time_val."""
        diff = np.abs(self.u_data['AlignedTime'].values - time_val)
        idx = int(np.argmin(diff))
        return float(self.u_data['U'].iloc[idx])


# ---------------------------------------------------------------------------


## Model and Material Parameters\n\nDefine trainable shift/Prony parameters, material constants, reduced-time factors, and constitutive helper properties.\n

In [ ]:
# 2. Trainable Shift Parameters
# ---------------------------------------------------------------------------

class InverseShiftParams(nn.Module):
    """
    Trainable time-temperature shift parameters for inverse identification.

    Trainable (3 DOF, independent log-parameterisation):
      log_C1:    WLF parameter C1  (init: log(8.0),     true: log(14.8))
      log_C2:    WLF parameter C2  (init: log(25.0),    true: log(45.6))
      log_Ea_R:  Arrhenius Ea/R    (init: log(20000.0), true: log(27403.3))

    Fixed:
      All elastic constants and Prony spectrum (from EX2 / UMAT ground truth)
      T_ref   = 323 K   (WLF reference temperature)
      T_arr   = 336 K   (Arrhenius reference temperature)
      T_cross = 317.4 K (WLF/Arrhenius crossover temperature)
    """

    # ---- Ground-truth values (for logging; not used in forward pass) ----
    TRUE_C1    = 14.8
    TRUE_C2    = 45.6
    TRUE_Ea_R  = 27403.3
    # True Prony weights [g0,g1,g2,g3,g4,g5,g_inf]; g2,g3,g4,g_inf are now identified.
    TRUE_G     = [0.206, 0.093, 0.306, 0.358, 0.034, 0.001, 0.002]
    TRUE_T_REF = 323.0
    TRUE_T_ARR = 336.0
    TRUE_T_CROSS = 317.4

    def __init__(self):
        super().__init__()

        # ------------------------------------------------------------------
        # Trainable shift parameters (log parameterisation → always positive)
        # Independent C1 and C2 — decoupled gradients avoid kappa-coupling drift.
        # Moderately low init: still non-trivial inverse, but avoids severe
        # early-stage drift and poor local minima in no-RF mode.
        # ------------------------------------------------------------------
        self.log_C1    = nn.Parameter(torch.tensor(math.log(8.0),     dtype=torch.float32))
        # C2 is held at its KNOWN value (45.6 K). The free-recovery experiment probes
        # a_T only over the narrow rubbery window (T≈325–353 K), so the WLF (C1,C2)
        # pair is fundamentally non-identifiable together — they trade off along a
        # diagonal valley (verified by a 2-D C1–C2 scan). C2 (the Vogel/free-volume
        # temperature, conventionally taken from literature/DMA) is therefore fixed;
        # the recovery then sharply identifies C1, and continuity gives Ea/R. This is
        # a structural identifiability resolution, NOT mid-training curriculum freezing.
        self.log_C2    = nn.Parameter(torch.tensor(math.log(45.6),    dtype=torch.float32))
        # Ea_R init: 20000 K (~73% of true 27403 K).
        # Previously 15000 caused near-zero gradient in ear_only phase (aT already saturated).
        # 20000 is still "wrong enough" to observe convergence but gives stronger gradient signal.
        self.log_Ea_R  = nn.Parameter(torch.tensor(math.log(20000.0), dtype=torch.float32))

        # ------------------------------------------------------------------
        # Fixed physical constants
        # ------------------------------------------------------------------
        self.T_ref  = 323.0   # WLF reference [K]
        self.T_arr  = 336.0   # Arrhenius reference [K]
        self.T_cross = 317.4  # crossover [K]

        # ------------------------------------------------------------------
        # Fixed Prony spectrum (6 branches). The slow ρ=10000 s branch is kept
        # SEPARATE (not merged into g_inf as in EX2). EX2's 900 s window could not
        # resolve it, but EX5's free RECOVERY is creep-controlled (~1/g_inf≈500×
        # slower than relaxation), so the 10000 s branch sets the recovery TIMING —
        # merging it into g_inf made the predicted recovery too fast and biased the
        # identified a_T (verified offline: with the slow branch the recovery loss is
        # minimised AT the true parameters, 4.9e-4 vs 5.2e-3 at the wrong optimum).
        # True: g0=0.206, g1=0.093, g2=0.306, g3=0.358, g4=0.034, g5=0.001, g_inf=0.002
        # ------------------------------------------------------------------
        self.N_prony = 6
        self.register_buffer("rho",
            torch.tensor([0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0], dtype=torch.float32))
        # ---- Prony spectrum (fixed in shift-only mode; trainable in audit mode) ----
        # g0(τ=0.1s), g1(τ=1s) and the slow g5(τ=10000s) branch are held FIXED:
        # g0/g1 are unidentifiable at the available Δt (same as EX2), and g5 pins the
        # free-recovery TIMING. The free weights {g2,g3,g4,g_inf} are identified; they
        # sum to G_free = 1-(g0+g1+g5) = 0.700, enforced by softmax×G_free so the full
        # 7-weight spectrum always normalises to 1. Wrong init (g3 -30%, g4 +135%,
        # g_inf +900%) keeps the joint inverse non-trivial and the convergence visible.
        self.register_buffer("g0_fixed", torch.tensor(0.206, dtype=torch.float32))
        self.register_buffer("g1_fixed", torch.tensor(0.093, dtype=torch.float32))
        self.register_buffer("g5_fixed", torch.tensor(0.001, dtype=torch.float32))
        self.G_free = 1.0 - (0.206 + 0.093 + 0.001)   # = 0.700
        # softmax(logits) ≈ [0.500,0.357,0.114,0.029] → g_free ≈ [0.350,0.250,0.080,0.020]
        self.logits_g_free = nn.Parameter(
            torch.tensor([-0.6931, -1.0296, -2.1716, -3.5539], dtype=torch.float32))

        # ------------------------------------------------------------------
        # Fixed elastic constants [MPa] (from True_Material_Parameters.md)
        # ------------------------------------------------------------------
        self.register_buffer("C11_0", torch.tensor(11250.16, dtype=torch.float32))
        self.register_buffer("C12_0", torch.tensor(891.26,   dtype=torch.float32))
        self.register_buffer("C22_0", torch.tensor(1425.85,  dtype=torch.float32))
        self.register_buffer("C23_0", torch.tensor(915.56,   dtype=torch.float32))
        self.register_buffer("C66_0", torch.tensor(267.69,   dtype=torch.float32))

        # Exact component-wise generalized-Maxwell constants from the EX5 Abaqus
        # *User Material table.  These must be used when the spectrum is prescribed:
        # a common scalar g_k is only an approximation because C11 has a distinct
        # fibre-dominated branch distribution.
        # Column order: C11, C12, C22, C23, C66; rows: tau=0.1...10000 s.
        self.register_buffer("C_branch_reference", torch.tensor([
            [200.254,   183.830, 293.707, 188.472, 55.3091],
            [ 89.0021,   82.8455,132.426,  84.9984,24.9132],
            [288.039,   272.998, 436.637, 280.342, 82.0325],
            [322.882,   318.639, 510.272, 327.818, 95.5597],
            [ 29.7953,   30.0832, 48.2092, 30.9813, 9.01153],
            [  1.08799,   1.0973,  1.75608, 1.1279, 0.328643],
        ], dtype=torch.float32))
        self.register_buffer("C_inf_reference", torch.tensor(
            [10319.1, 1.77138, 2.83844, 1.82402, 0.530688],
            dtype=torch.float32))
        self.use_exact_reference_spectrum = False

        # Long-term stiffness C∞
        # C11∞ is hardcoded from UMAT (fiber-dominated, different from matrix branches).
        # For matrix components (C12, C22, C23, C66) the true g_inf = 0.002, which now
        # equals g_weights[-1] (the 10000 s branch is a separate Prony term, no longer
        # merged into g_inf).
        # C11∞ is the fiber-dominated long-term stiffness (its own ~8% relaxation, set
        # by the UMAT; independent of the matrix g_inf) → kept as a fixed buffer.
        # The matrix long-term moduli C12/22/23/66∞ = g_inf·C_0 now follow the TRAINABLE
        # g_inf and are exposed as properties (C12_inf … below), not fixed buffers.
        self.register_buffer("C11_inf", 10319.10 * torch.ones(1, dtype=torch.float32))

        # 45° rotation buffer
        c = math.cos(math.radians(45.0))
        s = math.sin(math.radians(45.0))
        self.register_buffer("Q", torch.tensor([
            [ c, -s, 0.],
            [ s,  c, 0.],
            [ 0., 0., 1.]
        ], dtype=torch.float32))

    # ------------------------------------------------------------------ properties

    @property
    def C1(self):
        return torch.exp(self.log_C1)

    @property
    def C2(self):
        return torch.exp(self.log_C2)

    @property
    def Ea_R(self):
        return torch.exp(self.log_Ea_R)

    @property
    def g_matrix(self):
        """Full (7,) Prony weight vector [g0,g1,g2,g3,g4,g5,g_inf].

        g0,g1,g5 are fixed; {g2,g3,g4,g_inf} = softmax(logits)·G_free are trainable.
        The whole vector always sums to 1 (g0+g1+g5+G_free = 1).
        """
        g_free = torch.softmax(self.logits_g_free, dim=0) * self.G_free  # [g2,g3,g4,g_inf]
        return torch.cat([
            self.g0_fixed.view(1), self.g1_fixed.view(1),
            g_free[0:1], g_free[1:2], g_free[2:3],
            self.g5_fixed.view(1), g_free[3:4],
        ])

    @property
    def g_inf(self):
        """Trainable equilibrium fraction (last entry of g_matrix)."""
        return self.g_matrix[self.N_prony]

    # Matrix long-term moduli scale with the trainable g_inf (engineering shape (1,)).
    @property
    def C12_inf(self):
        return (self.g_inf * self.C12_0).unsqueeze(0)

    @property
    def C22_inf(self):
        return (self.g_inf * self.C22_0).unsqueeze(0)

    @property
    def C23_inf(self):
        return (self.g_inf * self.C23_0).unsqueeze(0)

    @property
    def C66_inf(self):
        return (self.g_inf * self.C66_0).unsqueeze(0)

    @torch.no_grad()
    def set_params(self, C1=None, C2=None, Ea_R=None):
        """Overwrite shift parameters in-place (used by the forward solver)."""
        if C1 is not None:
            self.log_C1.data.fill_(math.log(float(C1)))
        if C2 is not None:
            self.log_C2.data.fill_(math.log(float(C2)))
        if Ea_R is not None:
            self.log_Ea_R.data.fill_(math.log(float(Ea_R)))

    @torch.no_grad()
    def set_spectrum_to_true(self):
        """Reset the trainable Prony weights {g2,g3,g4,g_inf} to ground truth.

        Inverts softmax(logits)·G_free = g_free → logits = log(g_free / G_free).
        """
        tg = self.TRUE_G
        free = [tg[2], tg[3], tg[4], tg[6]]   # {g2,g3,g4,g_inf}, sum = G_free
        logits = [math.log(max(f, 1e-9) / self.G_free) for f in free]
        self.logits_g_free.data.copy_(
            torch.tensor(logits, dtype=torch.float32, device=self.logits_g_free.device))

    @torch.no_grad()
    def set_to_true(self):
        """Set all shift parameters AND the Prony spectrum to ground truth.

        Used by the FORWARD solver, which must reconstruct the field with the true
        material — so the spectrum (wrong-initialised for the inverse run) is reset here.
        """
        self.set_params(self.TRUE_C1, self.TRUE_C2, self.TRUE_Ea_R)
        self.set_spectrum_to_true()

    def shift_factor(self, T_tensor):
        """
        Hybrid WLF / Arrhenius shift factor a_T(T).

        Args:
            T_tensor: Temperature tensor, shape (..., 1) [K]

        Returns:
            a_T tensor of same shape (always > 0)
        """
        T = T_tensor
        C1 = self.C1
        C2 = self.C2
        Ea_R = self.Ea_R
        T_ref  = self.T_ref
        T_arr  = self.T_arr
        T_cross = self.T_cross

        # WLF (T > T_cross): log10(a_T) = -C1*(T-Tref)/(C2+(T-Tref))
        # Use explicit clamping + exp(log(10)*x) for better gradient stability.
        delta_T = T - T_ref
        denom = C2 + delta_T
        denom_safe = torch.clamp(denom, min=5.0, max=300.0)
        log10_aT_wlf = -C1 * delta_T / denom_safe
        log10_aT_wlf = torch.clamp(log10_aT_wlf, min=-12.0, max=12.0)
        aT_wlf = torch.exp(math.log(10.0) * log10_aT_wlf)

        # Arrhenius (T ≤ T_cross): a_T = exp(Ea_R*(1/T - 1/T_arr))
        T_safe = torch.clamp(T, min=200.0)  # avoid 1/T → inf
        arr_exponent = Ea_R * (1.0 / T_safe - 1.0 / T_arr)
        arr_exponent = torch.clamp(arr_exponent, min=-80.0, max=80.0)
        aT_arr = torch.exp(arr_exponent)

        # Blend smoothly around T_cross to avoid gradient spikes near the switch.
        blend = torch.sigmoid((T - T_cross) / 1.5)
        aT = blend * aT_wlf + (1.0 - blend) * aT_arr

        # Safety clamp: prevent extreme values that cause NaN in q recursion
        aT = torch.clamp(aT, min=1e-10, max=1e15)
        return aT

    def get_stiffness_matrix_inf(self):
        """Construct C^∞ stiffness matrix (6×6 Voigt)."""
        dev = self.C11_inf.device
        if self.use_exact_reference_spectrum:
            C11, C12, C22, C23, C66 = self.C_inf_reference
        else:
            C11 = self.C11_inf[0]
            C12 = self.C12_inf[0]
            C22 = self.C22_inf[0]
            C23 = self.C23_inf[0]
            C66 = self.C66_inf[0]
        C44 = (C22 - C23) / 2.0

        Cm = torch.zeros(6, 6, device=dev, dtype=torch.float32)
        Cm[0,0] = C11; Cm[1,1] = C22; Cm[2,2] = C22
        Cm[0,1] = C12; Cm[1,0] = C12
        Cm[0,2] = C12; Cm[2,0] = C12
        Cm[1,2] = C23; Cm[2,1] = C23
        Cm[3,3] = C66; Cm[4,4] = C66; Cm[5,5] = C44
        return Cm

    def get_stiffness_matrix_branch(self, k):
        """Construct Prony branch k stiffness matrix C^(k) (6×6 Voigt)."""
        dev = self.C11_0.device
        if self.use_exact_reference_spectrum:
            C11k, C12k, C22k, C23k, C66k = self.C_branch_reference[k]
        else:
            gk = self.g_matrix[k]
            # Six-parameter audit mode retains the shared-amplitude reduced
            # parameterisation; shift-only mode uses the exact table above.
            C11k = gk * (self.C11_0 - self.C11_inf[0])
            C12k = gk * self.C12_0
            C22k = gk * self.C22_0
            C23k = gk * self.C23_0
            C66k = gk * self.C66_0
        C44k = (C22k - C23k) / 2.0

        Ck = torch.zeros(6, 6, device=dev, dtype=torch.float32)
        Ck[0,0] = C11k; Ck[1,1] = C22k; Ck[2,2] = C22k
        Ck[0,1] = C12k; Ck[1,0] = C12k
        Ck[0,2] = C12k; Ck[2,0] = C12k
        Ck[1,2] = C23k; Ck[2,1] = C23k
        Ck[3,3] = C66k; Ck[4,4] = C66k; Ck[5,5] = C44k
        return Ck

    def compute_parameter_penalty(self):
        """Soft constraints to keep shift parameters in physical range."""
        dev = self.log_C1.device
        penalty = torch.zeros((), device=dev, dtype=torch.float32)
        C1 = self.C1
        C2 = self.C2
        Ea_R = self.Ea_R
        # C1 range [4, 30]
        penalty = penalty + 0.20 * torch.relu(4.0 - C1) ** 2
        penalty = penalty + 0.20 * torch.relu(C1 - 30.0) ** 2
        # C2 range [10, 100]
        penalty = penalty + 0.08 * torch.relu(10.0 - C2) ** 2
        penalty = penalty + 0.08 * torch.relu(C2 - 100.0) ** 2
        # Ea/R range [3000, 50000]
        penalty = penalty + 1e-5 * torch.relu(3000.0 - Ea_R) ** 2
        penalty = penalty + 1e-5 * torch.relu(Ea_R - 50000.0) ** 2
        # Physical continuity at crossover: aT_WLF(T_cross) = aT_Arrhenius(T_cross).
        # This equation links C1, C2, and Ea_R:
        #   -C1*(T_cross-Tref)/(C2+(T_cross-Tref)) = (Ea_R/ln10)*(1/T_cross-1/T_arr)
        # Weight raised from 1e-3 → 0.15 so the penalty (~7.5 at wrong init) is
        # comparable to other loss terms (~0.05-0.5), providing a strong identifiability
        # signal for Ea_R during ear_only phase instead of zero gradient.
        delta_cross = self.T_cross - self.T_ref
        denom_cross = torch.clamp(C2 + delta_cross, min=5.0, max=300.0)
        log10_a_wlf_cross = -C1 * delta_cross / denom_cross
        ln_a_wlf_cross = math.log(10.0) * log10_a_wlf_cross
        ln_a_arr_cross = Ea_R * (1.0 / self.T_cross - 1.0 / self.T_arr)
        penalty = penalty + 0.15 * (ln_a_wlf_cross - ln_a_arr_cross) ** 2
        return penalty

    def load_state_dict_compat(self, state_dict):
        """Load checkpoints in both kappa-based (legacy) and direct C1 (current) formats."""
        sd = dict(state_dict)
        if ('log_C1' not in sd) and ('log_kappa' in sd) and ('log_C2' in sd):
            # Legacy format: C1 = kappa * C2 → log_C1 = log_kappa + log_C2
            sd['log_C1'] = sd['log_kappa'] + sd['log_C2']
            sd.pop('log_kappa', None)
        current = self.state_dict()
        compatible = {}
        skipped = []
        for key, value in sd.items():
            if key not in current:
                skipped.append(key)
                continue
            if current[key].shape != value.shape:
                skipped.append(key)
                continue
            compatible[key] = value
        missing, unexpected = self.load_state_dict(compatible, strict=False)
        if skipped:
            print(f"  [compat] skipped incompatible material keys: {', '.join(skipped)}")
        if missing:
            print(f"  [compat] material keys kept at current defaults: {', '.join(missing)}")
        if unexpected:
            print(f"  [compat] unexpected material keys ignored: {', '.join(unexpected)}")


# ---------------------------------------------------------------------------


## Model Architecture\n\nDefine the spatiotemporal displacement network used by the RT-RPINN bending benchmark.\n

In [ ]:
# 3. Spatiotemporal PINN — 6D input (x,y,z, fiber coords, t, T)
# ---------------------------------------------------------------------------

class SpatiotemporalPINN(nn.Module):
    """
    Spatiotemporal PINN for non-isothermal viscoelastic displacement field.

    Input features:
      - raw normalised coords (x_hat, y_hat, z_hat, x_local, y_local, T_hat)   [6]
      - SPATIAL Fourier features sin/cos(2πk·{x,y,z}), k=1..n_fourier_space     [3*2*n_fs]
      - temporal Fourier features sin/cos(2πk·t), k=1..n_fourier, plus t        [2*n_fourier+1]

    Two changes vs the EX4-inherited backbone, required for EX5's LARGE-DEFORMATION
    bending (end rotation 4.71 rad ≈ 270°, displacements up to ~110 mm on a 95 mm beam):

      1. SPATIAL Fourier features. With raw coordinates only, a tanh MLP has strong
         spectral bias and collapses to a near-uniform displacement — it cannot form
         the steep gradient from u=0 (clamped end) to the large curl at the free end.
         Encoding x/y/z with sinusoids lets the network represent that variation.

      2. OUTPUT SCALING. The network predicts an O(1) field that is multiplied by
         `output_scale` (≈ max |u|). The raw layers then never need huge weights to
         reach ~110 mm, which dramatically improves conditioning/convergence.

    Output: (u1, u2, u3) displacement in mm.
    """

    def __init__(self, hidden=(128, 128, 128, 128), act=torch.tanh,
                 n_fourier=5, n_fourier_space=0, output_scale=1.0,
                 hard_left_bc=True):
        super().__init__()
        self.act = act
        self.n_fourier = n_fourier
        self.n_fourier_space = n_fourier_space
        self.hard_left_bc = hard_left_bc

        c = math.cos(math.radians(45.0))
        s = math.sin(math.radians(45.0))
        self.register_buffer("cos_theta", torch.tensor(c, dtype=torch.float32))
        self.register_buffer("sin_theta", torch.tensor(s, dtype=torch.float32))

        # Fourier frequencies for time and space encodings (2π·k)
        freqs = 2.0 * math.pi * torch.arange(1, n_fourier + 1, dtype=torch.float32)
        self.register_buffer("freqs", freqs)
        sfreqs = 2.0 * math.pi * torch.arange(1, n_fourier_space + 1, dtype=torch.float32)
        self.register_buffer("sfreqs", sfreqs)

        # Characteristic displacement scale (mm). Buffer → saved/restored with the model.
        self.register_buffer("output_scale",
                             torch.tensor(float(output_scale), dtype=torch.float32))

        # Input dim: 6 raw + 3*2*n_fs spatial Fourier + (2*n_fourier+1) temporal
        in_dim = 6 + 3 * 2 * n_fourier_space + 2 * n_fourier + 1
        dims = [in_dim, *hidden, 3]
        self.fcs = nn.ModuleList(
            [nn.Linear(dims[i], dims[i + 1]) for i in range(len(dims) - 1)]
        )
        # With a 125 mm output scale, the default Linear initialization creates
        # enormous displacement gradients before the first update. Start close to
        # the undeformed state; the data loss then grows the field smoothly.
        nn.init.normal_(self.fcs[-1].weight, mean=0.0, std=1.0e-4)
        nn.init.zeros_(self.fcs[-1].bias)

    def forward(self, x, y, z, t, T):
        """
        Args:
            x, y, z: Normalised spatial coords [0,1], shape (N,1)
            t: Normalised time [0,1], shape (N,1)
            T: Normalised temperature [0,1], shape (N,1)
        Returns:
            u: displacement (N,3) in mm
        """
        x_local = x * self.cos_theta + y * self.sin_theta
        y_local = -x * self.sin_theta + y * self.cos_theta

        # Spatial Fourier features for x, y, z
        sp_feats = []
        for v in (x, y, z):
            for f in self.sfreqs:
                sp_feats.append(torch.sin(v * f))
                sp_feats.append(torch.cos(v * f))

        # Temporal Fourier features
        t_enc = (
            [torch.sin(t * f) for f in self.freqs] +
            [torch.cos(t * f) for f in self.freqs] +
            [t]
        )

        h = torch.cat([x, y, z, x_local, y_local, T] + sp_feats + t_enc, dim=1)

        for layer in self.fcs[:-1]:
            h = self.act(layer(h))
        u = self.fcs[-1](h) * self.output_scale
        return x * u if self.hard_left_bc else u


# ---------------------------------------------------------------------------


## Solver and Loss Functions\n\nEvaluate finite-strain kinematics, recursive internal-variable evolution, stresses, equilibrium, boundary terms, and observable losses.\n

In [ ]:
# 4. Inverse PINN Solver
# ---------------------------------------------------------------------------

class InversePINNSolver:
    """
    Solver for inverse WLF/Arrhenius shift parameter identification.

    Physics:
      σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))
      q^(k)_n = exp(-Δξ_n/ρ_k)*q^(k)_{n-1} + (1 - exp(-Δξ_n/ρ_k))*ε_n
      Δξ_n = Δt_n / a_T(T_n)     ← key difference vs EX2

    All strain in MATERIAL coordinates (45° rotated).
    """

    def __init__(self, model, mat, fe_loader, bounds,
                 lambda_data=10.0, lambda_strain=10.0, lambda_stress=5.0,
                 lambda_pde=5.0,
                 lambda_bc_left=1.0, lambda_bc_right=10.0,
                 lambda_traction=5.0, lambda_rf=5.0, lambda_obs=20.0,
                 lambda_obs_rate=0.8, freeze_shift_params=False,
                 finite_strain=False, identify_from_fe_strain=False,
                 lambda_stress_rate=0.0, lambda_recovery=0.0, fix_c2=False,
                 lambda_jacobian=1.0, full_history_grad=True,
                 lambda_parameter_penalty=1.0, fix_prony=False):
        self.model = model.to(device)
        self.mat = mat.to(device)
        self.fe_loader = fe_loader
        self.bounds = bounds

        # Forward mode: hold the shift parameters fixed at their current values
        # (e.g. true/identified values) by zeroing their gradients every step.
        # This converts the inverse solver into a pure forward field reconstructor.
        self.freeze_shift_params = freeze_shift_params

        # Finite-strain (large-deformation) kinematics. When True, strain is the
        # objective Green-Lagrange measure E=½(FᵀF−I) (F=I+∂u/∂X) instead of the
        # small-strain symmetric gradient, the viscoelastic recursion runs on E
        # (Total-Lagrangian: q & 2nd-PK stress live in the reference frame), and the
        # constitutive 2nd-PK stress is pushed forward to Cauchy σ=J⁻¹FSFᵀ before it
        # is compared to the FE Cauchy stress. Required for EX5's ~270° rotation,
        # where small-strain theory reports huge spurious rotation strains.
        self.finite_strain = finite_strain

        # Identify shift parameters from the FE-MEASURED strain (Abaqus LE, an
        # objective co-rotational log strain) rather than from the network's own
        # differentiated displacement. Under large rotation the network gradient is
        # noisy and the finite-strain stress (∝ J⁻¹, J=detF) can explode, diverging
        # training; feeding the bounded measured strain into q(a_T)→σ and matching
        # the FE Cauchy stress gives a stable, well-posed identification decoupled
        # from the network. (No push-forward needed: LE & S are already reported in
        # the same co-rotational frame, exactly as in the small-strain EX2 inverse.)
        self.identify_from_fe_strain = identify_from_fe_strain
        self.lambda_jacobian = lambda_jacobian
        self.lambda_parameter_penalty = lambda_parameter_penalty
        self.fix_prony = bool(fix_prony)
        self.full_history_grad = full_history_grad
        self.mat._full_history_grad = bool(full_history_grad)

        self.lambda_data = lambda_data
        self.lambda_strain = lambda_strain
        self.lambda_stress = lambda_stress
        # Relaxation-rate term: matches dσ/dt between consecutive frames. a_T controls
        # how fast stress relaxes, so this sharpens the shift-parameter gradient.
        self.lambda_stress_rate = lambda_stress_rate
        # Recovery term: matches the normalised end-rotation (UR) recovery curve of
        # the free-recovery step (step 4). This is the PRIMARY, large, clean a_T
        # signal — the shape recovers as a_T drops on heating through Tg, so its
        # timing directly identifies the WLF/Arrhenius shift parameters. Fully
        # decoupled from the field network and from large-rotation kinematics.
        self.lambda_recovery = lambda_recovery
        # Hold C2 fixed at its known value (WLF C1↔C2 are non-identifiable together
        # from the recovery signal — see InverseShiftParams). C2 stays a constant for
        # the WHOLE run; only C1 and Ea/R are identified.
        self.fix_c2 = fix_c2
        self.lambda_pde = lambda_pde
        self.lambda_bc_left = lambda_bc_left
        self.lambda_bc_right = lambda_bc_right
        self.lambda_traction = lambda_traction
        self.lambda_rf = lambda_rf
        self.lambda_obs = lambda_obs
        self.lambda_obs_rate = lambda_obs_rate

        # Derived domain lengths
        self.x_min = bounds['x_min']; self.Lx = bounds['x_max'] - bounds['x_min']
        self.y_min = bounds['y_min']; self.Ly = bounds['y_max'] - bounds['y_min']
        self.z_min = bounds['z_min']; self.Lz = bounds['z_max'] - bounds['z_min']
        self.t_min = bounds['t_min']; self.T_max = bounds['t_max'] - bounds['t_min']
        self.Temp_min = bounds['T_min']
        self.Temp_range = max(bounds['T_max'] - bounds['T_min'], 1.0)

        # Abaqus material table: longitudinal/transverse thermal expansion.
        # The constitutive recursion is driven by mechanical Green--Lagrange
        # strain, while the FE displacement observations contain total motion.
        self.alpha_1 = 3.7e-6
        self.alpha_2 = 1.2e-4
        # EX5 input deck initializes the stress-free body at 343 K. This is
        # deliberately distinct from the 323 K WLF shift reference.
        self.T_ref_mech = 343.0

        self.loss_history = []
        self.loss_components_history = []
        self.param_history = []
        self.u_ref_sq = 1.0
        self.rf_rel_floor = torch.tensor(100.0, device=device)
        self.rf_node_ids = None
        self.rf_node_col = None
        # Free-recovery (step 4) UR-recovery target (filled by _prepare_recovery_target)
        self.rec_t = None         # free-recovery (step-4) frame times
        self.rec_T = None         # temperature at each recovery frame
        self.rec_dt = None        # real-time increments (→ reduced time via a_T)
        self.rec_R_fe = None      # FE UR-recovery target curve R_FE(t)

    # ------------------------------------------------------------------
    # Normalisation helpers
    # ------------------------------------------------------------------
    def norm_coords(self, x, y, z, t, T):
        x_hat = (x - self.x_min) / self.Lx
        y_hat = (y - self.y_min) / self.Ly
        z_hat = (z - self.z_min) / self.Lz
        t_hat = (t - self.t_min) / self.T_max
        T_hat = (T - self.Temp_min) / self.Temp_range
        return x_hat, y_hat, z_hat, t_hat, T_hat

    # ------------------------------------------------------------------
    # Rotation: global ↔ material coordinates (45° about Z)
    # ------------------------------------------------------------------
    def to_material_coords(self, strain_global):
        """Rotate engineering strain (N,6) from global to 45° material frame."""
        Q = self.mat.Q
        N = strain_global.shape[0]
        e11, e22, e33 = strain_global[:, 0], strain_global[:, 1], strain_global[:, 2]
        g12, g13, g23 = strain_global[:, 3], strain_global[:, 4], strain_global[:, 5]
        e12 = g12 / 2.0; e13 = g13 / 2.0; e23 = g23 / 2.0

        # Full strain tensor
        e = torch.stack([
            torch.stack([e11, e12, e13], dim=1),
            torch.stack([e12, e22, e23], dim=1),
            torch.stack([e13, e23, e33], dim=1),
        ], dim=2)  # (N,3,3)

        # Rotate: ε_mat = Q^T ε Q
        QT = Q.t()
        e_mat = torch.einsum('ij,njk,kl->nil', QT, e, Q)

        e11m = e_mat[:, 0, 0]; e22m = e_mat[:, 1, 1]; e33m = e_mat[:, 2, 2]
        e12m = e_mat[:, 0, 1]; e13m = e_mat[:, 0, 2]; e23m = e_mat[:, 1, 2]

        return torch.stack([e11m, e22m, e33m,
                            2.0*e12m, 2.0*e13m, 2.0*e23m], dim=1)

    def mechanical_material_strain(self, strain_global, temperature):
        """Rotate total strain to the material frame and subtract thermal strain."""
        strain_mat = self.to_material_coords(strain_global)
        dT = torch.as_tensor(temperature, dtype=strain_mat.dtype,
                             device=strain_mat.device) - self.T_ref_mech
        thermal = torch.zeros_like(strain_mat)
        thermal[:, 0] = self.alpha_1 * dT
        thermal[:, 1] = self.alpha_2 * dT
        thermal[:, 2] = self.alpha_2 * dT
        return strain_mat - thermal

    # ------------------------------------------------------------------
    # Strain via autodiff
    # ------------------------------------------------------------------
    def compute_strain(self, u, x_leaf, y_leaf, z_leaf):
        """Engineering strain (N,6) from displacement (N,3) via autodiff."""
        u1, u2, u3 = u[:, 0:1], u[:, 1:2], u[:, 2:3]

        def _grad(ui, xi):
            return torch.autograd.grad(ui, xi, torch.ones_like(ui),
                                       create_graph=True, retain_graph=True)[0]

        e11 = _grad(u1, x_leaf)
        e22 = _grad(u2, y_leaf)
        e33 = _grad(u3, z_leaf)
        g12 = _grad(u1, y_leaf) + _grad(u2, x_leaf)
        g13 = _grad(u1, z_leaf) + _grad(u3, x_leaf)
        g23 = _grad(u2, z_leaf) + _grad(u3, y_leaf)

        return torch.cat([e11, e22, e33, g12, g13, g23], dim=1)

    # ------------------------------------------------------------------
    # Finite-strain (large-deformation) kinematics
    # ------------------------------------------------------------------
    def compute_deformation_gradient(self, u, x_leaf, y_leaf, z_leaf):
        """Deformation gradient F = I + ∂u/∂X (N,3,3); X = reference coords.

        F[:,i,j] = δ_ij + ∂u_i/∂X_j. Reference coords are used because the CSV
        X,Y,Z are the (constant) reference positions and u is the displacement.
        """
        u1, u2, u3 = u[:, 0:1], u[:, 1:2], u[:, 2:3]

        def _grad(ui, xi):
            return torch.autograd.grad(ui, xi, torch.ones_like(ui),
                                       create_graph=True, retain_graph=True)[0]

        row1 = torch.cat([_grad(u1, x_leaf), _grad(u1, y_leaf), _grad(u1, z_leaf)], dim=1)
        row2 = torch.cat([_grad(u2, x_leaf), _grad(u2, y_leaf), _grad(u2, z_leaf)], dim=1)
        row3 = torch.cat([_grad(u3, x_leaf), _grad(u3, y_leaf), _grad(u3, z_leaf)], dim=1)
        H = torch.stack([row1, row2, row3], dim=1)  # (N,3,3), H[:,i,j]=∂u_i/∂X_j
        eye = torch.eye(3, device=H.device, dtype=H.dtype).unsqueeze(0)
        return eye + H

    @staticmethod
    def green_lagrange_6vec(F):
        """Green-Lagrange strain E=½(FᵀF−I) as an engineering 6-vector.

        Objective (zero under rigid rotation F=R). Engineering shear (×2 off-diag)
        matches the convention of compute_strain / to_material_coords.
        """
        C = torch.einsum('nki,nkj->nij', F, F)  # FᵀF (right Cauchy-Green)
        eye = torch.eye(3, device=F.device, dtype=F.dtype).unsqueeze(0)
        E = 0.5 * (C - eye)
        return torch.stack([E[:, 0, 0], E[:, 1, 1], E[:, 2, 2],
                            2.0 * E[:, 0, 1], 2.0 * E[:, 0, 2], 2.0 * E[:, 1, 2]], dim=1)

    @staticmethod
    def cauchy_from_pk2(S6, F):
        """Push 2nd-PK stress (global, 6-vector) forward to Cauchy: σ = J⁻¹ F S Fᵀ."""
        S = torch.stack([
            torch.stack([S6[:, 0], S6[:, 3], S6[:, 4]], dim=1),
            torch.stack([S6[:, 3], S6[:, 1], S6[:, 5]], dim=1),
            torch.stack([S6[:, 4], S6[:, 5], S6[:, 2]], dim=1),
        ], dim=1)  # (N,3,3) symmetric
        J = torch.det(F).abs().clamp(min=1e-6).unsqueeze(-1).unsqueeze(-1)
        sig = torch.einsum('nik,nkl,njl->nij', F, S, F) / J  # F S Fᵀ / J
        return torch.stack([sig[:, 0, 0], sig[:, 1, 1], sig[:, 2, 2],
                            sig[:, 0, 1], sig[:, 0, 2], sig[:, 1, 2]], dim=1)

    @staticmethod
    def pk1_from_pk2(S6, F):
        """First Piola stress P=F S for total-Lagrangian equilibrium."""
        S = torch.stack([
            torch.stack([S6[:, 0], S6[:, 3], S6[:, 4]], dim=1),
            torch.stack([S6[:, 3], S6[:, 1], S6[:, 5]], dim=1),
            torch.stack([S6[:, 4], S6[:, 5], S6[:, 2]], dim=1),
        ], dim=1)
        return torch.einsum('nij,njk->nik', F, S)

    # ------------------------------------------------------------------
    # Q recursion with reduced time (NON-ISOTHERMAL)
    # ------------------------------------------------------------------
    @staticmethod
    def compute_q_recursive(strain_mat_seq, dt_seq, T_seq, mat):
        """
        Internal-variable recursion with WLF reduced time.

        Args:
            strain_mat_seq: (N_t, N_pts, 6) material-frame strain
            dt_seq:         (N_t-1,) real-time increments [s]
            T_seq:          (N_t, N_pts) temperature at each time step [K]
            mat:            InverseShiftParams (provides shift_factor())

        Returns:
            q_seq: (N_t, N_pts, 6, N_prony)
        """
        N_t, N_pts, _ = strain_mat_seq.shape
        N_prony = mat.N_prony
        dev = strain_mat_seq.device

        q_list = [
            torch.zeros(N_pts, 6, N_prony, device=dev, dtype=strain_mat_seq.dtype)
        ]
        rho = mat.rho.view(1, 1, N_prony)  # (1,1,N_prony)
        truncate_every = None if getattr(mat, '_full_history_grad', False) else 50

        for n in range(1, N_t):
            dt = dt_seq[n - 1]  # scalar
            strain_n = strain_mat_seq[n]      # (N_pts, 6)
            strain_prev = strain_mat_seq[n - 1]  # (N_pts, 6)
            T_n = T_seq[n].reshape(N_pts, 1)  # (N_pts, 1)

            # Shift factor and reduced time increment
            aT_n = mat.shift_factor(T_n)           # (N_pts, 1)
            aT_n = torch.nan_to_num(aT_n, nan=1.0, posinf=1e12, neginf=1e-8)
            aT_n = torch.clamp(aT_n, min=1e-8, max=1e12)

            delta_xi = (dt / aT_n).unsqueeze(-1)   # (N_pts, 1, 1)
            delta_xi = torch.nan_to_num(delta_xi, nan=0.0, posinf=1e6, neginf=0.0)
            dxi_over_rho = delta_xi / rho          # (N_pts, 1, N_prony) via broadcast
            dxi_over_rho = torch.clamp(dxi_over_rho, min=0.0, max=80.0)

            exp_factor   = torch.exp(-dxi_over_rho)  # E = e^{-a}, (N_pts,1,N_prony)
            one_minus_E  = 1.0 - exp_factor          # 1 - E

            # Linear-strain (mid-point) exponential integrator: assume strain varies
            # LINEARLY across the step ε_{n-1}→ε_n and integrate dq/dξ=(ε-q)/ρ exactly.
            # q_n = E q_{n-1} + (1-φ) ε_n + (φ-E) ε_{n-1},  φ = (1-E)/a.
            # This stays accurate for large Δξ (sparse sampling), unlike the constant-
            # strain assumption q_n = E q_{n-1} + (1-E) ε_n, which drops the ε_{n-1}
            # term and mis-integrates the fast Prony branches (ρ=0.1/1 s) on the ramp.
            a = dxi_over_rho
            phi = torch.where(a > 1e-4,
                              one_minus_E / torch.clamp(a, min=1e-12),
                              1.0 - 0.5 * a)          # series limit (1-E)/a → 1-a/2
            coeff_n    = 1.0 - phi                     # weight on ε_n
            coeff_prev = phi - exp_factor             # weight on ε_{n-1}

            q_prev = q_list[-1]  # (N_pts, 6, N_prony)
            if truncate_every is not None and n % truncate_every == 0:
                q_prev = q_prev.detach()

            strain_n_safe    = torch.nan_to_num(strain_n,    nan=0.0, posinf=1e6, neginf=-1e6)
            strain_prev_safe = torch.nan_to_num(strain_prev, nan=0.0, posinf=1e6, neginf=-1e6)
            q_n = (exp_factor * q_prev
                   + coeff_n * strain_n_safe.unsqueeze(-1)
                   + coeff_prev * strain_prev_safe.unsqueeze(-1))
            q_n = torch.nan_to_num(q_n, nan=0.0, posinf=1e6, neginf=-1e6)
            q_list.append(q_n)

        return torch.stack(q_list, dim=0)

    # ------------------------------------------------------------------
    # Stress from strain and q (material coordinates → rotate to global)
    # ------------------------------------------------------------------
    def compute_stress(self, strain_mat, q_n, to_global=True):
        """
        Viscoelastic stress via the Prony branch sum.

        σ_mat = C^∞:ε_mat + Σ_k C^(k):(ε_mat - q^(k))
        σ_global = Q σ_mat Q^T   (only if to_global=True)

        Args:
            strain_mat: (N_pts, 6) engineering strain in material frame
            q_n:        (N_pts, 6, N_prony) internal variables in material frame
            to_global:  rotate material→global. Set False when comparing to FE
                        data already stored in the material (fiber) frame — Abaqus
                        outputs S/LE in the local material orientation (Ori-1).

        Returns:
            stress: (N_pts, 6) Voigt stress (global frame if to_global else material)
        """
        C_inf = self.mat.get_stiffness_matrix_inf()  # (6,6)
        sigma_mat = strain_mat @ C_inf.t()  # (N_pts, 6)

        for k in range(self.mat.N_prony):
            Ck = self.mat.get_stiffness_matrix_branch(k)  # (6,6)
            q_k = q_n[:, :, k]  # (N_pts, 6)
            sigma_mat = sigma_mat + (strain_mat - q_k) @ Ck.t()

        if not to_global:
            return sigma_mat

        # Rotate from material to global
        Q = self.mat.Q
        N = sigma_mat.shape[0]

        # Unpack Voigt (engineering convention)
        s11, s22, s33 = sigma_mat[:, 0], sigma_mat[:, 1], sigma_mat[:, 2]
        s12, s13, s23 = sigma_mat[:, 3], sigma_mat[:, 4], sigma_mat[:, 5]

        sig_t = torch.stack([
            torch.stack([s11, s12, s13], dim=1),
            torch.stack([s12, s22, s23], dim=1),
            torch.stack([s13, s23, s33], dim=1),
        ], dim=2)  # (N,3,3)

        sig_g = torch.einsum('ij,njk,kl->nil', Q, sig_t, Q.t())

        sg11 = sig_g[:, 0, 0]; sg22 = sig_g[:, 1, 1]; sg33 = sig_g[:, 2, 2]
        sg12 = sig_g[:, 0, 1]; sg13 = sig_g[:, 0, 2]; sg23 = sig_g[:, 1, 2]

        return torch.stack([sg11, sg22, sg33, sg12, sg13, sg23], dim=1)

    def right_face_q8_weights(self, coords_right):
        """Consistent nodal area weights for the 13x2 Q8 right-face mesh.

        Each 1x1 mm Q8 face contributes -A/12 at its four corner nodes and
        A/3 at its four midside nodes. Assembling all 26 faces integrates a
        constant exactly to the 26 mm^2 reference area.
        """
        weights = torch.zeros(coords_right.shape[0], dtype=coords_right.dtype,
                              device=coords_right.device)
        corner_c = -1.0 / 12.0
        midside_c = 1.0 / 3.0
        for iy in range(13):
            for iz in range(2):
                y0 = self.y_min + float(iy)
                z0 = self.z_min + float(iz)
                nodes = (
                    (y0,       z0,       corner_c),
                    (y0 + 1.0, z0,       corner_c),
                    (y0 + 1.0, z0 + 1.0, corner_c),
                    (y0,       z0 + 1.0, corner_c),
                    (y0 + 0.5, z0,       midside_c),
                    (y0 + 1.0, z0 + 0.5, midside_c),
                    (y0 + 0.5, z0 + 1.0, midside_c),
                    (y0,       z0 + 0.5, midside_c),
                )
                for yq, zq, coeff in nodes:
                    dist2 = ((coords_right[:, 1] - yq) ** 2
                             + (coords_right[:, 2] - zq) ** 2)
                    idx = torch.argmin(dist2)
                    if float(dist2[idx].detach().cpu()) > 1.0e-8:
                        raise RuntimeError(
                            f'Right-face Q8 node not found at (y,z)=({yq},{zq})'
                        )
                    weights[idx] = weights[idx] + coeff
        area = self.Ly * self.Lz
        if abs(float(weights.sum().detach().cpu()) - area) > 1.0e-5:
            raise RuntimeError('Right-face Q8 weights do not integrate to the face area')
        return weights

    def compute_reaction_moment_y(self, coords_right, u_right, P_right, weights):
        """Reaction moment about RP2's y-axis in the current configuration.

        P N is force per reference area, so the reference Q8 weights directly
        integrate force. The RP translation is the area-weighted current face
        centroid. The returned sign is the support reaction (opposite the moment
        exerted on the body); the training channel compares its magnitude because
        the exported EX5 RM history is stored as a positive magnitude.
        """
        area = weights.sum()
        x_cur = coords_right + u_right
        x_rp = torch.sum(weights.unsqueeze(1) * x_cur, dim=0) / area
        traction = P_right[:, :, 0]  # P N with reference outward N=e_X
        moment_density = torch.cross(x_cur - x_rp.unsqueeze(0), traction, dim=1)
        body_moment_y = torch.sum(weights * moment_density[:, 1])
        return -body_moment_y

    # ------------------------------------------------------------------
    # Free-recovery (step 4) UR-recovery signal — PRIMARY a_T identification
    # ------------------------------------------------------------------
    def _prepare_recovery_target(self, t_unload=71.0, t_end=121.0):
        """Precompute the normalised end-rotation (UR) recovery curve of step 4.

        R_FE(t) = UR(t)/UR(t_unload) for t in [t_unload, t_end], a scalar in [0,1]
        that decays from 1 (frozen shape just after unloading) to ~0 (recovered).
        Built from the dense full FE timeline + the UR history (NOT the subsampled
        field CSVs), so it is independent of spatial/temporal field subsampling and
        fully decoupled from the network. The matched prediction depends only on
        a_T(C1,C2,Ea/R) and the fixed Prony spectrum.
        """
        times = np.asarray(self.fe_loader.all_frame_times, dtype=float)
        mask = (times >= t_unload - 1e-6) & (times <= t_end + 1e-6)
        rec_t = np.sort(times[mask])
        if len(rec_t) < 5:
            print("  [Recovery] WARNING: <5 step-4 frames found; recovery loss disabled.")
            return
        ur = np.array([self.fe_loader.get_u_by_time(float(t)) for t in rec_t], dtype=float)
        ur0 = ur[0] if abs(ur[0]) > 1e-6 else (np.max(np.abs(ur)) + 1e-6)
        R_fe = ur / ur0
        Tfn = _STEP_T_FN[4]
        rec_T = np.clip(np.array([Tfn(float(t)) for t in rec_t]), 298.0, 353.0)
        rec_dt = np.diff(rec_t, prepend=rec_t[0])  # first increment = 0 (ξ starts at 0)
        self.rec_t    = torch.tensor(rec_t,  dtype=torch.float32, device=device)
        self.rec_T    = torch.tensor(rec_T,  dtype=torch.float32, device=device)
        self.rec_dt   = torch.tensor(rec_dt, dtype=torch.float32, device=device)
        self.rec_R_fe = torch.tensor(R_fe,   dtype=torch.float32, device=device)

        print(f"  [Recovery] {len(rec_t)} step-4 frames; R_FE {R_fe[0]:.3f}→{R_fe[-1]:.3f} "
              f"(UR {ur[0]:.3f}→{ur[-1]:.3f} rad, T {rec_T[0]:.0f}→{rec_T[-1]:.0f}K); "
              f"R(ξ;g) analytic — recovery now drives BOTH a_T AND the Prony spectrum.")

    def compute_recovery_loss(self):
        """Match the predicted free-recovery curve R_pred to the FE UR recovery.

        DIFFERENTIABLE-IN-SPECTRUM closed form. The frozen deformation stored in the
        Prony branches at unload releases under ZERO stress as the material heats
        through Tg. The un-recovered fraction R(ξ) solves the coupled creep-recovery
        system  dq_k/dξ = (ε - q_k)/ρ_k ,  ε = Σ g_k q_k /(g_inf + Σ g_k) , q_k(0)=1,
        a LINEAR system  dq/dξ = A q  with
            A_kj = g_j /(denom·ρ_k) − δ_kj/ρ_k ,   denom = g_inf + Σ g_k .
        Its modal solution gives, in closed form,
            R(ξ) = Σ_m C_m e^{λ_m ξ} / Σ_k g_k ,   C_m = (g·v_m)(w_m·q0) ,
        with (λ_m, v_m) the eigen-pairs of A and w_m the rows of V⁻¹. This is
        DIFFERENTIABLE w.r.t. BOTH the spectrum g (through A's eigen-pairs) AND
        a_T(C1,C2,Ea/R) (through ξ(t)=∫dt/a_T). The strong, clean recovery signal
        therefore identifies the Prony spectrum JOINTLY with the shift parameters —
        no longer leaning on the weak, a_T-independent relaxation-stress floor. The
        sharp drop where a_T collapses (T>Tg) pins C1; the frozen plateau pins Ea/R;
        the shape/timing of the release pins the spectral weights {g2,g3,g4,g_inf}.
        """
        if self.rec_t is None:
            return torch.zeros((), device=device)
        # Reduced time since unload (depends on the trainable a_T).
        aT = self.mat.shift_factor(self.rec_T.unsqueeze(1)).squeeze(1)   # (M,)
        aT = torch.clamp(torch.nan_to_num(aT, nan=1.0, posinf=1e15, neginf=1e-10),
                         min=1e-10, max=1e15)
        xi = torch.cumsum(self.rec_dt / aT, dim=0)                       # (M,)

        # Recovery system matrix A(g) — depends on the TRAINABLE spectrum.
        gm = self.mat.g_matrix                                           # (7,)
        g = gm[:self.mat.N_prony]                                        # (6,) branch weights
        g_inf = gm[self.mat.N_prony]
        rho = self.mat.rho                                              # (6,)
        denom = g_inf + g.sum()
        inv_rho = 1.0 / rho                                            # (6,)
        A = (g.unsqueeze(0) / denom) * inv_rho.unsqueeze(1) \
            - torch.diag(inv_rho)                                      # A[k,j]=g_j/(denom ρ_k)-δ/ρ_k

        # Eigen-decomposition → closed-form R(ξ). ρ are well separated and the coupling
        # (∝ g_inf/denom) is weak, so eigenvalues stay distinct → eig gradient is stable.
        lam, V = torch.linalg.eig(A)                                   # (6,), (6,6) complex
        Vinv = torch.linalg.inv(V)
        q0 = torch.ones(self.mat.N_prony, dtype=V.dtype, device=V.device)
        a_coef = Vinv @ q0                                             # modal coords of q0
        b_coef = g.to(V.dtype) @ V                                     # ε-projection per mode
        C = b_coef * a_coef                                           # (6,) modal weights of R
        denom_g = g.sum().to(V.dtype)                                  # Σ_m C_m = Σ_k g_k (real)

        # R(ξ_i) = Re[ Σ_m C_m exp(λ_m ξ_i) ] / Σ_k g_k.  Re(λ)<0 physically → clamp the
        # exponent's real part to [−60, 0]: ≤0 guards a numerically-positive eigenvalue,
        # ≥−60 avoids -inf underflow noise (exp(−60)≈0).
        expo = lam.unsqueeze(0) * xi.to(V.dtype).unsqueeze(1)         # (M,6) complex
        re = torch.clamp(expo.real, min=-60.0, max=0.0)
        expo = torch.complex(re, expo.imag)
        R_pred = (torch.exp(expo) * C.unsqueeze(0)).sum(dim=1).real / denom_g.real
        R_pred = torch.nan_to_num(R_pred, nan=1.0, posinf=1.0, neginf=0.0)
        return torch.mean((R_pred - self.rec_R_fe) ** 2)

    # ------------------------------------------------------------------
    # Initialise fixed RF node set (stable across epochs)
    # ------------------------------------------------------------------
    def initialize_fixed_rf_points(self, n_rf_points=512):
        full_data = self.fe_loader.full_data
        x_max = self.bounds['x_max']
        right_nodes = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]

        for col in ['NodeLabel', 'Node', 'NID']:
            if col in right_nodes.columns:
                self.rf_node_col = col
                node_ids = right_nodes[col].unique()
                np.random.shuffle(node_ids)
                self.rf_node_ids = node_ids[:n_rf_points]
                if self.lambda_rf > 0.0:
                    print(f"  RF fixed nodes: {len(self.rf_node_ids)} ({col})")
                else:
                    print(f"  Right-face fixed nodes: {len(self.rf_node_ids)} ({col}) [for obs/bc]")
                return
        self.rf_node_ids = None
        self.rf_node_col = None

    # ------------------------------------------------------------------
    # Temporal sequence sampler
    # ------------------------------------------------------------------
    def sample_temporal_sequence(self, full_data, batch_size, seq_length=60,
                                  required_node_ids=None, node_id_col=None):
        """
        Sample time subsequence covering all 4 cycle phases.

        Dense sampling at:
          (0,  4):    early programming ramp (τ=0.1s zone)
          (15, 25):   end of programming + start of cooling
          (20, 70):   cooling — shift factor changes rapidly (KEY for C1/C2)
          (65, 75):   end of cooling + unloading spring-back
          (71, 121):  recovery heating (KEY for Ea_R + final shape recovery)
        """
        if node_id_col is None:
            for col in ['NodeLabel', 'Node', 'NID']:
                if col in full_data.columns:
                    node_id_col = col
                    break
        use_node_ids = (node_id_col is not None) and (node_id_col in full_data.columns)

        unique_times = np.sort(full_data['Time'].unique())
        N_times = len(unique_times)
        seq_length = min(seq_length, N_times)

        if seq_length >= N_times:
            time_indices = np.arange(N_times)
        else:
            # Dense zones for non-isothermal cycle (weighted).
            # Cooling [20,70] carries the strongest C1/C2 signal, so assign
            # a larger fraction of temporal budget there.
            zone_specs = [
                ((0, 4), 0.7),       # ramp onset
                ((15, 25), 0.9),     # programming-end / cooling-start
                ((20, 70), 1.6),     # constrained cooling (broad WLF/Arrhenius)
                ((45, 65), 1.6),     # cooling crossover T≈317K — a_T observable (stacks on the broad zone)
                ((65, 75), 0.9),     # unloading transition
                ((71, 121), 1.4),    # recovery (broad)
                ((83, 103), 1.4),    # recovery Tg re-crossing — a_T observable
            ]
            fast_budget = max(10, int(round(0.70 * seq_length)))
            w_sum = sum(w for _, w in zone_specs)
            n_uniform = max(4, seq_length - fast_budget)

            selected = {0, N_times - 1}
            for (t_lo, t_hi), w in zone_specs:
                n_zone = max(2, int(round(fast_budget * w / w_sum)))
                zone_idx = np.where((unique_times >= t_lo) & (unique_times <= t_hi))[0]
                if len(zone_idx) > 0:
                    chosen = zone_idx[
                        np.round(np.linspace(0, len(zone_idx) - 1,
                                             n_zone)).astype(int)
                    ]
                    selected.update(chosen.tolist())

            uniform_all = np.round(np.linspace(0, N_times - 1, n_uniform)).astype(int)
            selected.update(uniform_all.tolist())

            time_indices = np.sort(np.array(list(selected), dtype=int))
            if len(time_indices) > seq_length:
                keep = np.round(np.linspace(0, len(time_indices) - 1,
                                            seq_length)).astype(int)
                time_indices = time_indices[keep]

        sampled_times = unique_times[time_indices]
        assert np.all(np.diff(sampled_times) >= 0), "sampled_times not monotonic"

        def _deduplicate_time_slice(df):
            """
            Abaqus step boundaries reuse absolute times (50/100/105 s), so
            a pure Time filter can return two frames with duplicate NodeLabel.
            Keep the later (Step, Frame) state for each node to obtain one
            stable nodal snapshot per absolute time.
            """
            if len(df) == 0:
                return df
            if node_id_col is None or node_id_col not in df.columns:
                return df
            sort_cols = [c for c in ['Step', 'Frame'] if c in df.columns]
            if sort_cols:
                df = df.sort_values(sort_cols)
            return df.drop_duplicates(subset=node_id_col, keep='last')

        # ------ Spatial region identification from first frame ------
        first_slice = full_data[
            np.isclose(full_data['Time'].values, sampled_times[0], atol=1e-5)
        ]
        first_slice = _deduplicate_time_slice(first_slice)
        ref_xyz_by_node = None
        if use_node_ids and node_id_col in first_slice.columns:
            ref_xyz_by_node = first_slice.set_index(node_id_col)[['X', 'Y', 'Z']]
        x_coords = first_slice['X'].values
        y_coords = first_slice['Y'].values
        z_coords = first_slice['Z'].values
        x_min = self.bounds['x_min']; x_max = self.bounds['x_max']
        y_min = self.bounds['y_min']; y_max = self.bounds['y_max']
        z_min = self.bounds['z_min']; z_max = self.bounds['z_max']
        tol = 1e-4

        is_left_face  = np.abs(x_coords - x_min) < tol
        is_right_face = np.abs(x_coords - x_max) < tol
        is_y_face = (np.abs(y_coords - y_min) < tol) | (np.abs(y_coords - y_max) < tol)
        is_z_face = (np.abs(z_coords - z_min) < tol) | (np.abs(z_coords - z_max) < tol)
        is_yz_face = is_y_face | is_z_face
        is_any_x = is_left_face | is_right_face
        is_interior = ~is_any_x & ~is_yz_face

        left_idx     = np.where(is_left_face)[0]
        right_idx    = np.where(is_right_face)[0]
        yz_idx       = np.where(is_yz_face & ~is_any_x)[0]
        interior_idx = np.where(is_interior)[0]

        n_left     = max(5,  batch_size // 12)
        n_right    = max(10, batch_size // 7)
        n_yz_faces = max(15, batch_size // 7)
        n_interior = batch_size - n_left - n_right - n_yz_faces

        n_left     = min(n_left,     len(left_idx))
        n_right    = min(n_right,    len(right_idx))
        n_yz_faces = min(n_yz_faces, len(yz_idx))
        n_interior = min(n_interior, len(interior_idx))

        # Fixed RF nodes (if provided)
        rf_fixed_active = False
        if required_node_ids is not None and use_node_ids and len(required_node_ids) > 0:
            first_right = first_slice.iloc[right_idx]
            mask_req = first_right[node_id_col].isin(required_node_ids)
            req_found = first_right[mask_req][node_id_col].values
            if len(req_found) > 0:
                rf_fixed_active = True
                if len(req_found) >= n_right:
                    # Align with EX2 stabilization: when fixed right-face IDs are
                    # available, use all of them for RF / right-BC supervision.
                    right_node_ids = req_found
                    n_right = len(right_node_ids)
                else:
                    extra = np.random.choice(
                        first_right[~mask_req][node_id_col].values,
                        max(0, n_right - len(req_found)), replace=False
                    )
                    right_node_ids = np.concatenate([req_found, extra])[:n_right]
            else:
                rr = np.random.choice(right_idx, n_right, replace=False) if n_right > 0 else []
                right_node_ids = first_slice.iloc[rr][node_id_col].values
        else:
            rr = np.random.choice(right_idx, n_right, replace=False) if n_right > 0 else []
            right_node_ids = first_slice.iloc[rr][node_id_col].values if use_node_ids else None

        # Sample other regions
        sl = np.random.choice(left_idx, n_left, replace=False) if n_left > 0 else []
        sy = np.random.choice(yz_idx, n_yz_faces, replace=False) if n_yz_faces > 0 else []
        si = np.random.choice(interior_idx, n_interior, replace=False) if n_interior > 0 else []

        left_node_ids = first_slice.iloc[sl][node_id_col].values if (use_node_ids and len(sl)) else None
        yz_node_ids   = first_slice.iloc[sy][node_id_col].values if (use_node_ids and len(sy)) else None
        interior_node_ids = first_slice.iloc[si][node_id_col].values if (use_node_ids and len(si)) else None

        # Build combined node ID order
        all_node_ids = np.concatenate([
            right_node_ids if right_node_ids is not None else [],
            left_node_ids if left_node_ids is not None else [],
            yz_node_ids if yz_node_ids is not None else [],
            interior_node_ids if interior_node_ids is not None else [],
        ])

        rf_end   = len(right_node_ids) if right_node_ids is not None else 0
        left_end = rf_end + (len(left_node_ids) if left_node_ids is not None else 0)
        yz_end   = left_end + (len(yz_node_ids) if yz_node_ids is not None else 0)

        # Collect data per time step
        coords_seq      = []
        u_data_seq      = []
        strain_data_seq = []
        stress_data_seq = []
        T_seq_list      = []
        frames          = []

        for t_val in sampled_times:
            time_slice = full_data[np.isclose(full_data['Time'].values, t_val, atol=1e-5)]
            time_slice = _deduplicate_time_slice(time_slice)
            if len(time_slice) == 0:
                continue

            # Node-ID-aligned reindex
            if use_node_ids and node_id_col in time_slice.columns:
                ts_indexed = time_slice.set_index(node_id_col)
                ts_aligned = ts_indexed.reindex(all_node_ids)
                if ref_xyz_by_node is not None:
                    ref_xyz = ref_xyz_by_node.reindex(all_node_ids)
                    for col in ['X', 'Y', 'Z']:
                        ts_aligned[col] = ts_aligned[col].fillna(ref_xyz[col])
            else:
                ts_aligned = time_slice.iloc[np.arange(min(len(all_node_ids), len(time_slice)))]

            xyz = torch.tensor(
                ts_aligned[['X', 'Y', 'Z']].fillna(0.0).values, dtype=torch.float32, device=device
            )
            u_vals = torch.tensor(
                ts_aligned[['U1', 'U2', 'U3']].fillna(0.0).values,
                dtype=torch.float32, device=device
            )
            T_val = float(time_slice['Temperature'].iloc[0])

            le_cols = ['LE11', 'LE22', 'LE33', 'LE12', 'LE13', 'LE23']
            if all(c in ts_aligned.columns for c in le_cols):
                strain_fe = torch.tensor(
                    ts_aligned[le_cols].fillna(0.0).values,
                    dtype=torch.float32, device=device
                )
                # Convert LE (tensor shear) to engineering shear (×2 for off-diagonal)
                strain_fe_eng = strain_fe.clone()
                strain_fe_eng[:, 3:] = strain_fe[:, 3:] * 2.0
                strain_data_seq.append(strain_fe_eng)
            else:
                strain_data_seq.append(None)

            # FE stress (global Cauchy) — direct shift-parameter gradient path
            s_cols = ['S11', 'S22', 'S33', 'S12', 'S13', 'S23']
            if all(c in ts_aligned.columns for c in s_cols):
                stress_fe = torch.tensor(
                    ts_aligned[s_cols].fillna(0.0).values,
                    dtype=torch.float32, device=device
                )
                stress_data_seq.append(stress_fe)
            else:
                stress_data_seq.append(None)

            coords_seq.append(xyz)
            u_data_seq.append(u_vals)
            T_seq_list.append(T_val)

            frame_val = int(time_slice['Frame'].iloc[0]) if 'Frame' in time_slice.columns else 0
            frames.append(frame_val)

        N_t = len(coords_seq)
        N_pts = len(all_node_ids)

        # dt sequence
        actual_times = [float(full_data[
            np.isclose(full_data['Time'].values, t_val, atol=1e-5)
        ]['Time'].iloc[0]) for t_val in sampled_times[:N_t]]

        dt_list = [actual_times[n] - actual_times[n-1] for n in range(1, N_t)]
        dt_seq = torch.tensor(dt_list, dtype=torch.float32, device=device)

        right_indices  = list(range(0, rf_end))
        left_indices   = list(range(rf_end, left_end))
        yz_face_indices = list(range(left_end, yz_end))
        interior_start  = yz_end

        return {
            'coords_seq':      coords_seq,
            'u_data_seq':      u_data_seq,
            'strain_data_seq': strain_data_seq,
            'stress_data_seq': stress_data_seq,
            'times':           actual_times[:N_t],
            'T_seq':           T_seq_list[:N_t],
            'dt_seq':          dt_seq,
            'frames':          frames[:N_t],
            'N_pts':           N_pts,
            'N_t':             N_t,
            'right_indices':   right_indices,
            'left_indices':    left_indices,
            'yz_face_indices': yz_face_indices,
            'interior_start':  interior_start,
            'n_right':         rf_end,
            'n_left':          left_end - rf_end,
            'n_yz_faces':      yz_end - left_end,
            'n_interior':      N_pts - yz_end,
        }

    # ------------------------------------------------------------------
    # Main training loop
    # ------------------------------------------------------------------
    def train(self, epochs=10000, batch_size=512,
              lr_network=2e-3, lr_params=2e-3,
              log_interval=100, seq_length=60,
              adaptive_loss_balance=True, adaptive_lr=True,
              field_block=2, material_block=8):

        start_time = time.time()
        full_data = self.fe_loader.full_data

        # Displacement reference scale
        x_max = self.bounds['x_max']
        right_data = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        u_mag = np.linalg.norm(right_data[['U1', 'U2', 'U3']].values, axis=1)
        u_ref = float(np.max(u_mag))
        self.u_ref_sq = (u_ref + 1e-10) ** 2

        rf_values = torch.tensor(self.fe_loader.rf_data['RF'].values,
                                  dtype=torch.float32, device=device)
        # Relative-error floor for the positive RM-magnitude history. It prevents
        # the near-zero ramp onset from dominating without suppressing the 2.8 N mm peak.
        rf_max_abs = float(torch.max(torch.abs(rf_values)).item()) if rf_values.numel() > 0 else 0.0
        rf_floor_val = max(0.10, 0.05 * rf_max_abs)
        self.rf_rel_floor = torch.tensor(rf_floor_val, dtype=torch.float32, device=device)
        self.rm_scale = torch.tensor(max(rf_max_abs, 0.10), dtype=torch.float32,
                                     device=device)
        print(f"\nU scale (max|u|): {u_ref:.4f} mm")
        if self.lambda_rf > 0.0:
            print(f"RM range: [{rf_values.min().item():.3f}, {rf_values.max().item():.3f}] N mm")
            print(f"RM loss scale: {self.rm_scale.item():.3f} N mm (smooth-L1)")
        else:
            print("RM supervision: disabled (lambda_rf=0)")

        # PRIMARY a_T signal: free-recovery (step 4) UR-recovery curve.
        if self.lambda_recovery > 0.0:
            self._prepare_recovery_target()

        # Align with EX2: use all available right-face nodes for RF supervision.
        x_max = self.bounds['x_max']
        right_nodes = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        n_rf_points = 512
        for col in ['NodeLabel', 'Node', 'NID']:
            if col in right_nodes.columns:
                n_rf_points = max(1, int(right_nodes[col].nunique()))
                break
        self.initialize_fixed_rf_points(n_rf_points=n_rf_points)

        for p in self.model.parameters():
            p.requires_grad_(True)
        for p in self.mat.parameters():
            p.requires_grad_(True)

        # Optimise only the parameters declared unknown for this inverse problem.
        # In the default bending shift-only mode, the EX5 reference spectrum and C2
        # are prescribed independently of EX2; only C1 and Ea/R enter this group.
        shift_params = [self.mat.log_C1, self.mat.log_Ea_R]
        if not self.fix_c2:
            shift_params.append(self.mat.log_C2)
        if not self.fix_prony and hasattr(self.mat, 'logits_g_free'):
            shift_params.append(self.mat.logits_g_free)
        if self.fix_c2:
            self.mat.log_C2.requires_grad_(False)
        if self.fix_prony and hasattr(self.mat, 'logits_g_free'):
            self.mat.logits_g_free.requires_grad_(False)
        adapt_log_vars = {}
        adapt_param_group = []
        if adaptive_loss_balance:
            # Keep stress term at a fixed explicit weight: it provides the most
            # direct gradient path to shift parameters (C1/C2/Ea_R).
            # Only create adaptive weights for ACTIVE loss channels (lambda>0) so
            # disabled terms (e.g. rf/obs/pde in EX5) don't clutter the logs.
            _lambda_of = {
                "data": self.lambda_data, "strain": self.lambda_strain,
                "obs": self.lambda_obs, "obs_rate": self.lambda_obs_rate,
                "rf": self.lambda_rf, "pde": self.lambda_pde,
            }
            for name, lam in _lambda_of.items():
                if lam <= 0.0:
                    continue
                p = nn.Parameter(torch.tensor(0.0, dtype=torch.float32, device=device))
                adapt_log_vars[name] = p
                adapt_param_group.append(p)

        opt_groups = [
            {'params': self.model.parameters(), 'lr': lr_network},
            {'params': shift_params,            'lr': lr_params},
        ]
        if len(adapt_param_group) > 0:
            opt_groups.append({'params': adapt_param_group, 'lr': min(1e-3, lr_params)})

        no_rf_mode = self.lambda_rf <= 0.0

        optimizer = Adam(opt_groups)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
        # Keep shift LR from collapsing too early; otherwise C1/C2/EaR freeze
        # in a wrong local minimum while the field network keeps fitting data.
        shift_lr_floor = max(6e-5, 0.25 * lr_params)
        net_lr_floor = max(1e-5, 0.05 * lr_network)
        # Adaptive-LR monitor state (plateau-triggered base-LR reduction).
        best_id_monitor = float('inf')
        plateau_count = 0
        plateau_patience = max(120, epochs // 25)
        id_improve_tol = 1e-3
        lr_reduce_net = 0.85
        lr_reduce_shift = 0.90
        lr_reduce_adapt = 0.85

        print(f"\nStarting EX5 inverse training ({epochs} epochs)...")
        print(f"  seq_length={seq_length}, batch_size={batch_size}")
        print(f"  adaptive_loss_balance={adaptive_loss_balance} adaptive_lr={adaptive_lr}")
        print(f"  Initial C1={self.mat.C1.item():.2f} (true {self.mat.TRUE_C1})")
        print(f"  Initial C2={self.mat.C2.item():.2f} (true {self.mat.TRUE_C2})")
        print(f"  Initial Ea_R={self.mat.Ea_R.item():.1f} (true {self.mat.TRUE_Ea_R})")
        print("-" * 80)

        def _clamp_shift_params():
            if hasattr(self.mat, 'log_C1'):
                self.mat.log_C1.data = torch.nan_to_num(
                    self.mat.log_C1.data, nan=math.log(10.0),
                    posinf=math.log(30.0), neginf=math.log(4.0)
                )
                self.mat.log_C1.data.clamp_(math.log(4.0), math.log(30.0))
            if hasattr(self.mat, 'log_C2'):
                self.mat.log_C2.data = torch.nan_to_num(
                    self.mat.log_C2.data, nan=math.log(30.0),
                    posinf=math.log(100.0), neginf=math.log(10.0)
                )
                self.mat.log_C2.data.clamp_(math.log(10.0), math.log(100.0))
            if hasattr(self.mat, 'log_Ea_R'):
                self.mat.log_Ea_R.data = torch.nan_to_num(
                    self.mat.log_Ea_R.data, nan=math.log(20000.0),
                    posinf=math.log(50000.0), neginf=math.log(3000.0)
                )
                self.mat.log_Ea_R.data.clamp_(math.log(3000.0), math.log(50000.0))

        def _sanitize_gradients(params):
            all_finite = True
            n_bad_tensors = 0
            for p in params:
                if p.grad is None:
                    continue
                finite = torch.isfinite(p.grad)
                if not finite.all():
                    all_finite = False
                    n_bad_tensors += 1
                    p.grad.data = torch.nan_to_num(p.grad.data, nan=0.0, posinf=0.0, neginf=0.0)
            return all_finite, n_bad_tensors

        # Alternating field/material optimisation.  A simultaneous update lets the
        # flexible displacement network change its gradients while the material
        # parameters change, so the two blocks can compensate each other even when
        # RM, equilibrium and displacement data are all matched.  Alternation keeps
        # the current material fixed during a field step and the current field fixed
        # during a material step.  This uses no parameter target or external case.
        field_block = max(1, int(field_block))
        material_block = max(1, int(material_block))
        alternating_period = field_block + material_block
        warmup_epochs = min(1000, max(100, epochs // 10))
        physics_ramp_epochs = max(50, warmup_epochs // 2)
        print(f"  Optimisation: field warm-up for {warmup_epochs} epochs, then "
              f"a {physics_ramp_epochs}-epoch TL-physics ramp")
        print(f"  Alternating blocks: field={field_block}, material={material_block} epoch(s)")

        # Observation-window monitor scores (diagnostic only). These are NOT
        # used to override the trained parameters: the wlf/ear residual proxy
        # does not correlate with parameter accuracy, so restoring an earlier
        # "best" snapshot rolled the shift parameters away from their converged
        # (near-truth) end-of-training values. We now always keep the
        # trained-end parameters and only log these scores.
        best_wlf_score = float('inf')
        best_ear_score = float('inf')
        prev_phase = None

        def _set_phase_requires_grad(epoch):
            # FORWARD mode: shift params are held fixed (frozen grads) for a clean
            # full-field reconstruction.
            if self.freeze_shift_params:
                for p in self.model.parameters():
                    p.requires_grad_(True)
                for attr in ('log_C1', 'log_C2', 'log_Ea_R'):
                    p = getattr(self.mat, attr, None)
                    if p is not None:
                        p.requires_grad_(False)
                if hasattr(self.mat, 'logits_g_free'):
                    self.mat.logits_g_free.requires_grad_(False)
                return "field_warmup"
            if epoch < warmup_epochs:
                for p in self.model.parameters():
                    p.requires_grad_(True)
                for attr in ('log_C1', 'log_C2', 'log_Ea_R', 'logits_g_free'):
                    p = getattr(self.mat, attr, None)
                    if p is not None:
                        p.requires_grad_(False)
                for p in adapt_log_vars.values():
                    p.requires_grad_(True)
                return "field_warmup"

            block_pos = (epoch - warmup_epochs) % alternating_period
            material_phase = block_pos >= field_block
            for p in self.model.parameters():
                p.requires_grad_(not material_phase)
            for attr in ('log_C1', 'log_C2', 'log_Ea_R'):
                p = getattr(self.mat, attr, None)
                if p is not None:
                    p.requires_grad_(material_phase)
            if hasattr(self.mat, 'logits_g_free'):
                self.mat.logits_g_free.requires_grad_(material_phase and not self.fix_prony)
            if self.fix_c2 and hasattr(self.mat, 'log_C2'):
                self.mat.log_C2.requires_grad_(False)
            # Do not let uncertainty weights absorb the residual during a material
            # step; they are learned only with the field representation.
            for p in adapt_log_vars.values():
                p.requires_grad_(not material_phase)
            return "material_tl" if material_phase else "field_tl"

        for epoch in range(epochs):
            phase = _set_phase_requires_grad(epoch)
            if epoch < warmup_epochs:
                physics_scale = 0.0
            else:
                physics_scale = min(
                    1.0, float(epoch - warmup_epochs + 1) / float(physics_ramp_epochs)
                )
            if prev_phase != phase:
                print(f"[Epoch {epoch:5d}] Phase transition: {prev_phase} → {phase}")
            prev_phase = phase
            self.model.train()
            optimizer.zero_grad()

            seq_data = self.sample_temporal_sequence(
                full_data,
                batch_size=batch_size,
                seq_length=seq_length,
                required_node_ids=self.rf_node_ids,
                node_id_col=self.rf_node_col,
            )

            coords_seq      = seq_data['coords_seq']
            u_data_seq      = seq_data['u_data_seq']
            strain_data_seq = seq_data['strain_data_seq']
            stress_data_seq = seq_data['stress_data_seq']
            sampled_times   = seq_data['times']
            T_seq           = seq_data['T_seq']
            dt_seq          = seq_data['dt_seq']
            N_pts = seq_data['N_pts']
            N_t   = seq_data['N_t']
            rf_indices      = seq_data['right_indices']
            left_indices    = seq_data['left_indices']
            yz_face_indices = seq_data['yz_face_indices']
            interior_start  = seq_data['interior_start']

            # ---- Single forward pass for all time steps ----
            strain_seq      = []
            u_pred_seq      = []
            coord_grads_seq = []
            F_seq           = []   # deformation gradients (finite-strain mode only)

            for n in range(N_t):
                coords = coords_seq[n]
                t_val  = sampled_times[n]
                T_val  = T_seq[n]
                t_tensor = torch.full((N_pts, 1), t_val, dtype=torch.float32, device=device)
                T_tensor = torch.full((N_pts, 1), T_val, dtype=torch.float32, device=device)

                x_c = coords[:, 0:1].clone().requires_grad_(True)
                y_c = coords[:, 1:2].clone().requires_grad_(True)
                z_c = coords[:, 2:3].clone().requires_grad_(True)

                xh, yh, zh, th, Th = self.norm_coords(x_c, y_c, z_c, t_tensor, T_tensor)
                u_pred = self.model(xh, yh, zh, th, Th)
                if self.finite_strain:
                    # Objective Green-Lagrange strain from the deformation gradient.
                    F_n = self.compute_deformation_gradient(u_pred, x_c, y_c, z_c)
                    strain = self.green_lagrange_6vec(F_n)
                    F_seq.append(F_n)
                else:
                    strain = self.compute_strain(u_pred, x_c, y_c, z_c)
                    F_seq.append(None)

                u_pred_seq.append(u_pred)
                strain_seq.append(strain)
                coord_grads_seq.append((x_c, y_c, z_c))

            # ---- Objective mechanical strain in the material frame ----
            strain_mat_seq = []
            for n in range(N_t):
                strain_mat_seq.append(
                    self.mechanical_material_strain(strain_seq[n], T_seq[n])
                )
            strain_mat_tensor = torch.stack(strain_mat_seq, dim=0)  # (N_t, N_pts, 6)

            # Temperature tensor for q recursion: (N_t, N_pts)
            T_for_q = torch.tensor(T_seq, dtype=torch.float32, device=device)  # (N_t,)
            T_for_q = T_for_q.unsqueeze(1).expand(N_t, N_pts)  # (N_t, N_pts)

            # ---- Q recursion with reduced time ----
            q_seq = self.compute_q_recursive(strain_mat_tensor, dt_seq, T_for_q, self.mat)

            # ---- FE-strain constitutive q (robust shift-parameter identification) ----
            # q computed from the MEASURED strain history → its only dependence on the
            # trainable parameters is through a_T(C1,C2,Ea_R) in the reduced time, so
            # matching the resulting stress to the FE stress identifies the shift
            # parameters without going through the network's noisy displacement gradient.
            # FE LE/S are stored in the MATERIAL (fiber) frame (Abaqus local
            # orientation Ori-1), so use them directly — no to_material_coords (that
            # would double-rotate) and compare the material-frame stress (to_global=False).
            q_fe_seq = None
            strain_fe_mat_seq = None
            if (self.identify_from_fe_strain and self.lambda_stress > 0.0
                    and all(s is not None for s in strain_data_seq)):
                strain_fe_mat_seq = [strain_data_seq[n] for n in range(N_t)]
                strain_fe_mat_tensor = torch.stack(strain_fe_mat_seq, dim=0)
                q_fe_seq = self.compute_q_recursive(strain_fe_mat_tensor, dt_seq,
                                                    T_for_q, self.mat)

            rf_indices_torch = torch.tensor(rf_indices, dtype=torch.long, device=device)
            if rf_indices_torch.numel() == 0:
                print(f"[Epoch {epoch:5d}] RF node set is empty. right_indices={rf_indices}")
                break
            rm_weights = self.right_face_q8_weights(
                coords_seq[0][rf_indices_torch, :]
            ) if self.lambda_rf > 0.0 else None

            # ---- Accumulate losses ----
            loss_data_total     = torch.zeros((), device=device)
            loss_strain_total   = torch.zeros((), device=device)
            loss_stress_total   = torch.zeros((), device=device)
            loss_stress_rate_total = torch.zeros((), device=device)
            n_stress_rate       = 0
            prev_stress_pred = None; prev_stress_fe = None; prev_t_stress = None
            loss_rf_total       = torch.zeros((), device=device)
            n_rf_terms          = 0
            rm_pred_epoch       = []
            rm_target_epoch     = []
            loss_obs_total      = torch.zeros((), device=device)
            loss_obs_rate_total = torch.zeros((), device=device)
            loss_bc_left_total  = torch.zeros((), device=device)
            loss_bc_right_total = torch.zeros((), device=device)
            loss_traction_total = torch.zeros((), device=device)
            loss_pde_total      = torch.zeros((), device=device)
            loss_jacobian_total = torch.zeros((), device=device)
            prev_u_obs_pred = None
            prev_u_obs_target = None
            prev_t_obs = None
            n_obs_rate_terms = 0
            # Observation-only monitors for phase-best snapshots
            rf_wlf_sum = 0.0
            rf_wlf_terms = 0
            obs_wlf_sum = 0.0
            obs_wlf_terms = 0
            rf_ear_sum = 0.0
            rf_ear_terms = 0
            obs_ear_sum = 0.0
            obs_ear_terms = 0
            obs_rate_ear_sum = 0.0
            obs_rate_ear_terms = 0
            pde_sample_interval = 30
            if phase in ("ear_only", "wlf_only", "c1c2_refine", "ear_refine"):
                pde_sample_interval = 15
            phase_rf_scale = 1.0
            if phase in ("ear_only", "ear_refine"):
                phase_rf_scale = 0.40 if not no_rf_mode else 0.25
            elif phase in ("wlf_only", "c1c2_refine"):
                phase_rf_scale = 1.20 if not no_rf_mode else 1.2
            # During parameter-identification phases, slightly downweight strain
            # to avoid it overwhelming RF/observable signals.
            phase_strain_scale = 1.0
            if phase in ("ear_only", "wlf_only"):
                phase_strain_scale = 0.50 if no_rf_mode else 0.35
            elif phase in ("c1c2_refine", "ear_refine"):
                phase_strain_scale = 0.42 if no_rf_mode else 0.30
            elif phase == "joint_finetune":
                phase_strain_scale = 0.60 if no_rf_mode else 0.40

            for n in range(N_t):
                coords   = coords_seq[n]
                u_data   = u_data_seq[n]
                u_pred   = u_pred_seq[n]
                strain_n = strain_seq[n]   # global frame
                q_n      = q_seq[n]        # material frame
                t_val = sampled_times[n]
                T_val = float(T_seq[n])
                # Temperature-window gating:
                # - WLF phases focus on high-T segment
                # - Ea phases focus on low-T segment
                phase_temp_scale = 1.0
                if phase in ("wlf_only", "c1c2_refine"):
                    if T_val >= 320.0:
                        phase_temp_scale = 1.6
                    elif T_val <= 315.0:
                        phase_temp_scale = 0.20
                    else:
                        phase_temp_scale = 0.90
                elif phase in ("ear_only", "ear_refine"):
                    if T_val <= 315.0:
                        phase_temp_scale = 1.8
                    elif T_val >= 320.0:
                        phase_temp_scale = 0.20
                    else:
                        phase_temp_scale = 0.90
                elif phase == "joint_finetune":
                    # Mild focus around cooling/high-T windows to protect C1/C2
                    # from drifting during late all-parameter updates.
                    if T_val >= 320.0:
                        phase_temp_scale = 1.15
                    elif T_val <= 310.0:
                        phase_temp_scale = 0.85
                    else:
                        phase_temp_scale = 1.0
                # Time-window gating (stronger than temperature-only gating):
                # - WLF phases: focus constrained cooling [20,70] s
                # - Ea phases : focus free-recovery [71,121] s
                phase_time_scale = 1.0
                if phase in ("wlf_only", "c1c2_refine"):
                    if 20.0 <= t_val < _T_CROSS_TIME:
                        phase_time_scale = 1.8   # WLF cooling zone: C1/C2 signal
                    elif _T_CROSS_TIME <= t_val < 70.0:
                        phase_time_scale = 0.40  # Arrhenius zone: suppress for C1/C2
                    elif t_val < 20.0:
                        phase_time_scale = 0.35
                    else:
                        phase_time_scale = 0.20
                elif phase in ("ear_only", "ear_refine"):
                    if t_val >= 71.0:
                        phase_time_scale = 1.8   # free recovery: main Ea/R signal
                    elif t_val >= 70.0:
                        phase_time_scale = 1.0   # spring-back
                    elif t_val >= _T_CROSS_TIME:
                        phase_time_scale = 1.5   # Arrhenius cooling (48-70s): Ea/R signal
                    elif t_val >= 20.0:
                        phase_time_scale = 0.20  # WLF cooling zone: suppress
                    else:
                        phase_time_scale = 0.10
                elif phase == "joint_finetune":
                    # Keep a light cooling emphasis in late stage so WLF params
                    # remain anchored by the most informative segment.
                    if 20.0 <= t_val <= 70.0:
                        phase_time_scale = 1.20
                    elif t_val < 20.0:
                        phase_time_scale = 0.75
                    elif t_val >= 71.0:
                        phase_time_scale = 1.05
                    else:
                        phase_time_scale = 0.90
                phase_focus_scale = phase_temp_scale * phase_time_scale

                # Data loss
                loss_data_total += phase_focus_scale * (torch.mean((u_pred - u_data) ** 2) / self.u_ref_sq)

                # The FE output is logarithmic strain, whereas this formulation uses
                # Green--Lagrange strain. It is diagnostic only and must not enter
                # the total-Lagrangian inverse objective.
                if self.lambda_strain > 0.0 and strain_data_seq[n] is not None:
                    strain_fe = strain_data_seq[n]
                    eps_ref_sq = 0.11 ** 2  # max |LE| ≈10.3% for EX5 bending
                    loss_strain_total += phase_focus_scale * torch.mean(
                        (strain_n - strain_fe) ** 2
                    ) / eps_ref_sq

                # Stress for RF / traction / PDE / stress-data loss
                strain_mat_n = strain_mat_seq[n]
                # Constitutive output is second Piola stress in the reference frame.
                S_n = self.compute_stress(strain_mat_n, q_n)
                if self.finite_strain and F_seq[n] is not None:
                    P_n = self.pk1_from_pk2(S_n, F_seq[n])
                    stress_n = (self.cauchy_from_pk2(S_n, F_seq[n])
                                if self.lambda_stress > 0.0
                                else S_n)
                    J_n = torch.det(F_seq[n])
                    # Reject orientation reversal without perturbing valid states.
                    loss_jacobian_total += torch.mean(torch.relu(0.20 - J_n) ** 2)
                else:
                    P_n = None
                    stress_n = S_n

                # Stress-data loss — direct dense gradient path to shift parameters.
                # sigma_pred depends on q(a_T(C1,C2,Ea_R), strain), so matching
                # FE stress uniquely constrains the shift parameters. Prefer the
                # FE-measured-strain stress (stable); fall back to the network stress.
                if self.lambda_stress > 0.0 and stress_data_seq[n] is not None:
                    sigma_ref_sq = 50.0 ** 2  # 50 MPa normalisation
                    if q_fe_seq is not None:
                        # Material-frame stress (FE S is in the fiber frame).
                        stress_pred = self.compute_stress(strain_fe_mat_seq[n],
                                                          q_fe_seq[n], to_global=False)
                    else:
                        stress_pred = stress_n
                    # a_T-sensitivity window weight: emphasise the times where the
                    # shift factor changes fastest — constrained cooling (WLF→Arrhenius
                    # crossover at ~48 s) and the recovery re-crossing of Tg (~88 s) —
                    # to sharpen C1/C2/Ea_R identifiability.
                    # Concentrate weight where a_T is actually OBSERVABLE: the
                    # WLF→Arrhenius crossover (reduced time sweeps through the Prony
                    # window). High-T (saturated, fully relaxed) and deep-cold
                    # (frozen) segments carry little a_T information, so down-weight.
                    w_aT = 1.0
                    if 45.0 <= t_val < 65.0:        # cooling crossover (T≈317K @48.4s) — sharpest signal
                        w_aT = 2.2
                    elif 20.0 <= t_val < 45.0:       # early cooling (WLF, still informative)
                        w_aT = 1.3
                    elif 65.0 <= t_val < 70.0:       # late cooling (Arrhenius, near-frozen)
                        w_aT = 1.2
                    elif 85.0 <= t_val <= 100.0:     # recovery Tg re-crossing (T≈317K @88.6s)
                        w_aT = 2.0
                    loss_stress_total += w_aT * torch.mean(
                        (stress_pred - stress_data_seq[n]) ** 2
                    ) / sigma_ref_sq

                    # Relaxation-increment term: match the stress CHANGE Δσ between
                    # consecutive frames. This is the CLEAN a_T signal: subtracting
                    # consecutive frames CANCELS the a_T-independent static model-
                    # mismatch offset (fixed Prony/elastic ≠ full UMAT) that pins the
                    # absolute-stress loss at an irreducible ~3e-3 floor and biases its
                    # optimum. Δσ is ~10× smaller than σ, so it gets a dedicated, smaller
                    # normalisation (10 MPa) — otherwise the 50 MPa scale makes this term
                    # ~25× too small and the static floor drowns the relaxation signal.
                    if (self.lambda_stress_rate > 0.0 and prev_stress_pred is not None
                            and prev_t_stress is not None):
                        sigma_rate_ref_sq = 10.0 ** 2  # Δσ ~10 MPa scale
                        dsig_pred = stress_pred - prev_stress_pred
                        dsig_fe = stress_data_seq[n] - prev_stress_fe
                        loss_stress_rate_total += w_aT * torch.mean(
                            (dsig_pred - dsig_fe) ** 2
                        ) / sigma_rate_ref_sq
                        n_stress_rate += 1
                    prev_stress_pred = stress_pred
                    prev_stress_fe = stress_data_seq[n]
                    prev_t_stress = t_val

                # PDE loss (sampled)
                compute_pde = (physics_scale > 0.0) and (self.lambda_pde > 0.0) \
                    and (n % pde_sample_interval == 0) and (interior_start < N_pts)
                if compute_pde:
                    x_c_n, y_c_n, z_c_n = coord_grads_seq[n]
                    if P_n is None:
                        raise RuntimeError('EX5 TL inverse requires finite_strain=True')
                    P_pde = P_n[interior_start:, :, :]

                    def _pde_grad(s, leaf):
                        g = torch.autograd.grad(s, leaf, torch.ones_like(s),
                                                create_graph=True, allow_unused=True)[0]
                        return g[interior_start:] if g is not None else torch.zeros_like(s)

                    # Div_X P = 0: derivative over the reference-coordinate index j.
                    div_x = (_pde_grad(P_pde[:, 0, 0:1], x_c_n)
                             + _pde_grad(P_pde[:, 0, 1:2], y_c_n)
                             + _pde_grad(P_pde[:, 0, 2:3], z_c_n))
                    div_y = (_pde_grad(P_pde[:, 1, 0:1], x_c_n)
                             + _pde_grad(P_pde[:, 1, 1:2], y_c_n)
                             + _pde_grad(P_pde[:, 1, 2:3], z_c_n))
                    div_z = (_pde_grad(P_pde[:, 2, 0:1], x_c_n)
                             + _pde_grad(P_pde[:, 2, 1:2], y_c_n)
                             + _pde_grad(P_pde[:, 2, 2:3], z_c_n))
                    pde_res = torch.cat([div_x, div_y, div_z], dim=1)

                    # Use the material stress scale and the smallest geometric
                    # length. The earlier 100 MPa / 95 mm scale over-weighted the
                    # thin-direction derivatives by more than five orders overall.
                    L_ref = min(self.Lx, self.Ly, self.Lz)
                    pde_ref_sq = (1000.0 / L_ref) ** 2
                    loss_pde_total += torch.mean(torch.sum(pde_res ** 2, dim=1)) / pde_ref_sq

                # Reaction-moment magnitude at constrained RP2 (steps 1--2).
                # The released stages are governed by P N=0 and carry no RM target.
                if physics_scale > 0.0 and self.lambda_rf > 0.0 and t_val < 70.0:
                    P_right = P_n[rf_indices_torch, :, :]
                    rm_signed = self.compute_reaction_moment_y(
                        coords[rf_indices_torch, :],
                        u_pred[rf_indices_torch, :],
                        P_right,
                        rm_weights,
                    )
                    rm_pred = torch.sqrt(rm_signed ** 2 + 1.0e-12)
                    rm_target = torch.tensor(
                        abs(self.fe_loader.get_rf_by_time(t_val)),
                        dtype=torch.float32, device=device
                    )
                    if (not torch.isfinite(rm_pred)) or (not torch.isfinite(rm_target)):
                        print(f"[Epoch {epoch:5d}] Non-finite RM at time {t_val:.6f}s")
                        loss_rf_total = torch.tensor(float('nan'), device=device)
                        break
                    rm_residual = (rm_pred - rm_target) / self.rm_scale
                    rm_loss_n = Fnn.smooth_l1_loss(
                        rm_residual, torch.zeros_like(rm_residual),
                        beta=0.25, reduction='mean'
                    )
                    loss_rf_total += rm_loss_n
                    n_rf_terms += 1
                    rm_pred_epoch.append(float(rm_pred.detach().cpu()))
                    rm_target_epoch.append(float(rm_target.detach().cpu()))
                    rm_monitor_val = float(torch.clamp(rm_loss_n.detach(), max=1e6).item())
                    if 20.0 <= t_val < _T_CROSS_TIME:
                        rf_wlf_sum += rm_monitor_val
                        rf_wlf_terms += 1
                    if _T_CROSS_TIME <= t_val < 70.0:
                        rf_ear_sum += rm_monitor_val
                        rf_ear_terms += 1

                # Observable loss on loaded-end kinematics history.
                # NOTE (EX5): the scalar end observable is the ROTATION UR (rad),
                # which is incompatible with the network's displacement output, so
                # this term is disabled by default (lambda_obs=0). It is guarded so
                # the rotation target never pollutes the loss/monitors. Full-field
                # data carries the recovery signal that drives shift identification.
                if self.lambda_obs > 0.0:
                    u_obs_pred = u_pred_seq[n][rf_indices_torch, 0].mean()
                    u_obs_target = torch.tensor(
                        self.fe_loader.get_u_by_time(t_val),
                        dtype=torch.float32, device=device
                    )
                    obs_weight = 1.0
                    if t_val >= 70.0:
                        obs_weight = 5.0
                    elif t_val >= 20.0:
                        obs_weight = 2.0
                    if abs(t_val - 70.0) <= 5.0 or abs(t_val - 71.0) <= 5.0 or abs(t_val - _T_END) <= 5.0:
                        obs_weight *= 2.0
                    # In no-RM mode, emphasize phase-sensitive time windows to improve
                    # identifiability separation:
                    # - WLF phases focus on cooling/constrained segment (20~70 s)
                    # - Ea phases focus on free recovery segment (>=71 s)
                    phase_obs_scale = 1.0
                    if phase in ("wlf_only", "c1c2_refine"):
                        if 20.0 <= t_val <= 70.0:
                            phase_obs_scale = 3.0 if no_rf_mode else 2.0
                        elif t_val < 20.0:
                            phase_obs_scale = 0.35 if no_rf_mode else 0.70
                        else:
                            phase_obs_scale = 0.50 if no_rf_mode else 0.80
                    elif phase in ("ear_only", "ear_refine"):
                        if t_val >= 71.0:
                            phase_obs_scale = 3.0 if no_rf_mode else 2.5
                        elif t_val >= 20.0:
                            phase_obs_scale = 1.2 if no_rf_mode else 0.9
                        else:
                            phase_obs_scale = 0.30 if no_rf_mode else 0.60
                    obs_rel_sq = ((u_obs_pred - u_obs_target) ** 2) / self.u_ref_sq
                    loss_obs_total += (obs_weight * phase_obs_scale * phase_focus_scale) * obs_rel_sq
                    obs_rel_sq_val = float(torch.clamp(obs_rel_sq.detach(), max=1e6).item())
                    if 20.0 <= t_val < 70.0:
                        obs_wlf_sum += obs_rel_sq_val
                        obs_wlf_terms += 1
                    if t_val >= 71.0:
                        obs_ear_sum += obs_rel_sq_val
                        obs_ear_terms += 1

                    # Observable-rate loss (d/dt) on loaded-end kinematics history.
                    # This term is especially informative for Ea/R during free recovery.
                    if prev_u_obs_pred is not None and prev_t_obs is not None:
                        dt_obs = max(t_val - prev_t_obs, 1e-6)
                        u_rate_pred = (u_obs_pred - prev_u_obs_pred) / dt_obs
                        u_rate_target = (u_obs_target - prev_u_obs_target) / dt_obs
                        # Only apply rate supervision near unloading/recovery.
                        obs_rate_scale = 0.0
                        if t_val >= 71.0:
                            obs_rate_scale = 1.0
                        elif t_val >= 70.0:
                            obs_rate_scale = 0.5
                        if phase in ("ear_only", "ear_refine"):
                            obs_rate_scale *= 1.10
                        elif phase in ("wlf_only", "c1c2_refine"):
                            obs_rate_scale *= 0.30
                        if obs_rate_scale > 0.0:
                            # Relative rate error with floor avoids scale explosion.
                            rate_denom = torch.clamp(torch.abs(u_rate_target), min=0.08)
                            rate_res_sq = torch.clamp(((u_rate_pred - u_rate_target) / rate_denom) ** 2, max=25.0)
                            loss_obs_rate_total += (obs_rate_scale * phase_focus_scale) * rate_res_sq
                            if t_val >= 71.0:
                                obs_rate_ear_sum += float(torch.clamp(rate_res_sq.detach(), max=1e6).item())
                                obs_rate_ear_terms += 1
                            n_obs_rate_terms += 1

                    prev_u_obs_pred = u_obs_pred
                    prev_u_obs_target = u_obs_target
                    prev_t_obs = t_val

                # BC right (prescribed-rotation end: match FE displacement until release)
                if t_val < 70.0:
                    u_bc_right_pred   = u_pred_seq[n][rf_indices_torch, :]
                    u_bc_right_target = u_data_seq[n][rf_indices_torch, :]
                    loss_bc_right_total += torch.mean(
                        (u_bc_right_pred - u_bc_right_target) ** 2
                    ) / self.u_ref_sq
                else:
                    # After unloading (t >= 70 s), the right end is released.
                    # Enforce traction-free condition on the right face to keep
                    # recovery-stage mechanics coupled to shift parameters.
                    tx = P_n[rf_indices_torch, :, 0]  # P N, N=e_X
                    sigma_ref_sq = 1000.0 ** 2
                    free_w = 2.0 if t_val >= 71.0 else 1.0
                    loss_bc_right_total += free_w * torch.mean(tx ** 2) / sigma_ref_sq

                # BC left (fixed, u=0)
                if len(left_indices) > 0:
                    li = torch.tensor(left_indices, dtype=torch.long, device=device)
                    loss_bc_left_total += torch.mean(
                        u_pred_seq[n][li, :] ** 2
                    ) / self.u_ref_sq

                # Traction-free lateral BC
                if len(yz_face_indices) > 0:
                    yi = torch.tensor(yz_face_indices, dtype=torch.long, device=device)
                    coords_yz = coords[yi, :]
                    sigma_ref_sq = 1000.0 ** 2
                    y_min = self.bounds['y_min']; y_max = self.bounds['y_max']
                    z_min = self.bounds['z_min']; z_max = self.bounds['z_max']
                    tol = 1e-4
                    is_y = (torch.abs(coords_yz[:, 1] - y_min) < tol) | \
                           (torch.abs(coords_yz[:, 1] - y_max) < tol)
                    is_z = (torch.abs(coords_yz[:, 2] - z_min) < tol) | \
                           (torch.abs(coords_yz[:, 2] - z_max) < tol)
                    if is_y.any():
                        ty = P_n[yi[is_y], :, 1]  # P N, N=+/-e_Y; sign vanishes in MSE
                        loss_traction_total += torch.mean(ty ** 2) / sigma_ref_sq
                    if is_z.any():
                        tz = P_n[yi[is_z], :, 2]  # P N, N=+/-e_Z
                        loss_traction_total += torch.mean(tz ** 2) / sigma_ref_sq

            # Average over time steps
            loss_data     = loss_data_total     / N_t
            loss_strain   = phase_strain_scale * (loss_strain_total / N_t)
            loss_stress   = loss_stress_total   / N_t
            loss_stress_rate = loss_stress_rate_total / max(1, n_stress_rate)
            loss_rf       = loss_rf_total       / max(1, n_rf_terms)
            loss_obs      = loss_obs_total      / N_t
            loss_obs_rate = loss_obs_rate_total / max(1, n_obs_rate_terms)
            loss_bc_left  = loss_bc_left_total  / N_t
            loss_bc_right = loss_bc_right_total / N_t
            loss_traction = loss_traction_total / N_t
            n_pde_computed = max(1, sum(1 for i in range(N_t)
                                        if i % pde_sample_interval == 0))
            loss_pde = loss_pde_total / n_pde_computed
            loss_jacobian = loss_jacobian_total / N_t

            loss_penalty = self.mat.compute_parameter_penalty()
            # Phase-specific identification monitors (observation windows only).
            wlf_monitor = float('inf')
            if rf_wlf_terms > 0 or obs_wlf_terms > 0:
                wlf_monitor = 0.0
                if rf_wlf_terms > 0:
                    wlf_monitor += rf_wlf_sum / rf_wlf_terms
                if obs_wlf_terms > 0:
                    wlf_monitor += 0.25 * (obs_wlf_sum / obs_wlf_terms)
            ear_monitor = float('inf')
            if rf_ear_terms > 0 or obs_ear_terms > 0 or obs_rate_ear_terms > 0:
                ear_monitor = 0.0
                if obs_ear_terms > 0:
                    ear_monitor += obs_ear_sum / obs_ear_terms
                if obs_rate_ear_terms > 0:
                    ear_monitor += 0.80 * (obs_rate_ear_sum / obs_rate_ear_terms)
                if rf_ear_terms > 0:
                    ear_monitor += 0.20 * (rf_ear_sum / rf_ear_terms)
            # Track best observation-window monitor scores (diagnostic only;
            # no longer used to override the trained shift parameters).
            if phase == "material_tl" and np.isfinite(wlf_monitor):
                best_wlf_score = min(best_wlf_score, wlf_monitor)
            if phase == "material_tl" and np.isfinite(ear_monitor):
                best_ear_score = min(best_ear_score, ear_monitor)

            # Uniform loss weights — with stress-data loss providing the primary
            # shift-parameter gradient, phase-specific reweighting is not needed.
            # Displacement data update the representation only.  During a material
            # step they are constant with respect to the active variables and are
            # omitted from the reported/optimised objective for clarity.
            phase_w_data = 0.0 if phase == "material_tl" else 1.0
            phase_w_strain = 1.0
            phase_w_obs = 1.0
            phase_w_obs_rate = 1.0
            phase_w_rf = 1.0
            phase_w_pde = 1.0

            base_terms = {
                'data':     (phase_w_data * self.lambda_data) * loss_data,
                'strain':   (phase_w_strain * self.lambda_strain) * loss_strain,
                'obs':      (phase_w_obs * self.lambda_obs) * loss_obs,
                'obs_rate': (phase_w_obs_rate * self.lambda_obs_rate) * loss_obs_rate,
                'rf':       physics_scale * (phase_w_rf * self.lambda_rf) * loss_rf,
                'pde':      physics_scale * (phase_w_pde * self.lambda_pde) * loss_pde,
            }
            loss_anchor = torch.zeros((), device=device)

            # PRIMARY a_T signal: free-recovery (step 4) UR-recovery curve.
            # Depends only on a_T(C1,C2,Ea/R) + fixed Prony → strong, clean gradient
            # to the shift parameters, decoupled from the (saturated) stress match.
            loss_recovery = (self.compute_recovery_loss()
                             if self.lambda_recovery > 0.0
                             else torch.zeros((), device=device))

            loss_total = (
                physics_scale * self.lambda_bc_left  * loss_bc_left +
                physics_scale * self.lambda_bc_right * loss_bc_right +
                physics_scale * self.lambda_traction * loss_traction +
                self.lambda_stress * loss_stress +
                self.lambda_stress_rate * loss_stress_rate +
                (self.lambda_recovery * loss_recovery
                 if phase == "material_tl" else 0.0 * loss_recovery) +
                physics_scale * self.lambda_jacobian * loss_jacobian +
                (self.lambda_parameter_penalty * loss_penalty
                 if phase == "material_tl" else 0.0 * loss_penalty) +
                loss_anchor
            )
            if adaptive_loss_balance and len(adapt_log_vars) > 0:
                for name, term in base_terms.items():
                    coeff = float(term.detach().abs().item())
                    if coeff <= 0.0:
                        loss_total = loss_total + term
                        continue
                    lv = adapt_log_vars[name]
                    # Heteroscedastic uncertainty weighting:
                    # exp(-lv) * L + reg(lv). This auto-downweights exploding terms.
                    loss_total = loss_total + torch.exp(-lv) * term + 0.05 * (lv ** 2)
            else:
                for term in base_terms.values():
                    loss_total = loss_total + term

            if not torch.isfinite(loss_total):
                print(f"[Epoch {epoch:5d}] Non-finite loss detected.")
                print(f"  finite(data)={torch.isfinite(loss_data).item()} "
                      f"finite(strain)={torch.isfinite(loss_strain).item()} "
                      f"finite(rf)={torch.isfinite(loss_rf).item()} "
                      f"finite(obs)={torch.isfinite(loss_obs).item()} "
                      f"finite(pde)={torch.isfinite(loss_pde).item()} "
                      f"finite(penalty)={torch.isfinite(loss_penalty).item()}")
                print(f"  C1={self.mat.C1.item()} C2={self.mat.C2.item()} Ea_R={self.mat.Ea_R.item()}")
                break

            loss_total.backward()

            # Forward mode: keep shift parameters fixed by zeroing their gradients
            # every step, regardless of phase/requires_grad toggling.
            if self.freeze_shift_params:
                for attr in ('log_C1', 'log_C2', 'log_Ea_R', 'logits_g_free'):
                    p = getattr(self.mat, attr, None)
                    if p is not None and p.grad is not None:
                        p.grad.zero_()

            # In warmup phase, zero any shift/spectrum gradients that leaked
            # through requires_grad_(False) (safety guard only).
            if phase == "field_warmup":
                for attr in ('log_C1', 'log_C2', 'log_Ea_R', 'logits_g_free'):
                    p = getattr(self.mat, attr, None)
                    if p is not None and p.grad is not None:
                        p.grad.zero_()

            # C2 held fixed at its known value (non-identifiable): zero any leaked grad.
            if self.fix_c2 and hasattr(self.mat, 'log_C2') and self.mat.log_C2.grad is not None:
                self.mat.log_C2.grad.zero_()

            c1_bad = int(hasattr(self.mat, 'log_C1') and self.mat.log_C1.grad is not None
                         and (not torch.isfinite(self.mat.log_C1.grad).all()))
            c2_bad = int(hasattr(self.mat, 'log_C2') and self.mat.log_C2.grad is not None
                         and (not torch.isfinite(self.mat.log_C2.grad).all()))
            ear_bad = int(hasattr(self.mat, 'log_Ea_R') and self.mat.log_Ea_R.grad is not None
                          and (not torch.isfinite(self.mat.log_Ea_R.grad).all()))

            grads_finite_model, n_bad_model = _sanitize_gradients(self.model.parameters())
            grads_finite_shift, n_bad_shift = _sanitize_gradients(shift_params)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(shift_params, max_norm=0.8)

            if not (grads_finite_model and grads_finite_shift):
                print(f"[Epoch {epoch:5d}] Non-finite gradients detected; sanitizing and continuing.")
                print(f"  bad_model_tensors={n_bad_model} bad_shift_tensors={n_bad_shift} "
                      f"bad_logC1={c1_bad} bad_logC2={c2_bad} bad_logEaR={ear_bad}")

            optimizer.step()
            scheduler.step()
            if adaptive_loss_balance and len(adapt_log_vars) > 0:
                for p in adapt_log_vars.values():
                    p.data.clamp_(-3.0, 3.0)
            # Enforce a practical lower bound for shift-parameter LR.
            if optimizer.param_groups[1]['lr'] < shift_lr_floor:
                optimizer.param_groups[1]['lr'] = shift_lr_floor
            if optimizer.param_groups[0]['lr'] < net_lr_floor:
                optimizer.param_groups[0]['lr'] = net_lr_floor

            # Count an identification plateau only when the shift parameters are
            # actually active.  In the EX5 pilot the old all-epoch counter reduced
            # the shift LR during the 100-epoch field warm-up, before C1 or Ea/R had
            # received a single update.
            if adaptive_lr and phase == "material_tl":
                # Monitor only active identification channels. In particular,
                # exclude PDE when lambda_pde=0 to avoid irrelevant LR decay.
                id_monitor = (
                    loss_obs.detach().item() +
                    0.6 * loss_obs_rate.detach().item()
                )
                # The free-recovery curve is the primary a_T signal; track it most.
                if self.lambda_recovery > 0.0:
                    id_monitor += 10.0 * loss_recovery.detach().item()
                # Track the relaxation-increment more heavily than the absolute
                # stress, which sits at an a_T-independent floor.
                if self.lambda_stress_rate > 0.0:
                    id_monitor += 4.0 * loss_stress_rate.detach().item()
                if self.lambda_stress > 0.0:
                    id_monitor += 0.3 * loss_stress.detach().item()
                if self.lambda_rf > 0.0:
                    id_monitor += 0.6 * loss_rf.detach().item()
                if self.lambda_pde > 0.0:
                    id_monitor += 0.4 * loss_pde.detach().item()
                if id_monitor < best_id_monitor * (1.0 - id_improve_tol):
                    best_id_monitor = id_monitor
                    plateau_count = 0
                else:
                    plateau_count += 1
                    if plateau_count >= plateau_patience:
                        # Reduce cosine base_lrs so future scheduler values also drop.
                        scheduler.base_lrs[0] = max(net_lr_floor, scheduler.base_lrs[0] * lr_reduce_net)
                        scheduler.base_lrs[1] = max(shift_lr_floor, scheduler.base_lrs[1] * lr_reduce_shift)
                        if len(scheduler.base_lrs) > 2:
                            scheduler.base_lrs[2] = max(1e-5, scheduler.base_lrs[2] * lr_reduce_adapt)
                        optimizer.param_groups[0]['lr'] = max(net_lr_floor, optimizer.param_groups[0]['lr'] * lr_reduce_net)
                        optimizer.param_groups[1]['lr'] = max(shift_lr_floor, optimizer.param_groups[1]['lr'] * lr_reduce_shift)
                        if len(optimizer.param_groups) > 2:
                            optimizer.param_groups[2]['lr'] = max(1e-5, optimizer.param_groups[2]['lr'] * lr_reduce_adapt)
                        plateau_count = 0
                        print(
                            f"[Epoch {epoch:5d}] adaptive-lr reduce: "
                            f"net={optimizer.param_groups[0]['lr']:.2e} "
                            f"shift={optimizer.param_groups[1]['lr']:.2e} "
                            f"id_monitor={id_monitor:.4e}"
                        )
            _clamp_shift_params()

            # Store history
            self.loss_history.append(loss_total.item())
            rm_pred_mean = (float(np.mean(rm_pred_epoch))
                            if rm_pred_epoch else float('nan'))
            rm_target_mean = (float(np.mean(rm_target_epoch))
                              if rm_target_epoch else float('nan'))
            self.loss_components_history.append({
                'data':     loss_data.item(),
                'strain':   loss_strain.item(),
                'stress':   loss_stress.item(),
                'recovery': loss_recovery.item(),
                'obs':      loss_obs.item(),
                'obs_rate': loss_obs_rate.item(),
                'bc_left':  loss_bc_left.item(),
                'bc_right': loss_bc_right.item(),
                'traction': loss_traction.item(),
                'rf':       loss_rf.item(),
                'rm_pred_mean': rm_pred_mean,
                'rm_target_mean': rm_target_mean,
                'pde':      loss_pde.item(),
                'jacobian': loss_jacobian.item(),
                'penalty':  loss_penalty.item(),
                'anchor':   loss_anchor.item(),
            })
            _gm_now = self.mat.g_matrix.detach().cpu().tolist()
            self.param_history.append({
                'C1':   self.mat.C1.item(),
                'C2':   self.mat.C2.item(),
                'Ea_R': self.mat.Ea_R.item(),
                'g2':    _gm_now[2],
                'g3':    _gm_now[3],
                'g4':    _gm_now[4],
                'g_inf': _gm_now[6],
            })

            if epoch % log_interval == 0 or epoch == epochs - 1:
                C1_now   = self.mat.C1.item()
                C2_now   = self.mat.C2.item()
                EaR_now  = self.mat.Ea_R.item()
                lr_net   = optimizer.param_groups[0]['lr']
                lr_sft   = optimizer.param_groups[1]['lr']
                t_range  = f"[{sampled_times[0]:.1f},{sampled_times[-1]:.1f}]s"
                T_min_seq = min(T_seq)
                T_max_seq = max(T_seq)
                T_range  = f"[{T_min_seq:.0f},{T_max_seq:.0f}]K"
                # Only print loss channels that are active (lambda>0). In the forward
                # data-fit, strain/stress are off (small-strain measure is invalid
                # under large rotation) so they are hidden to avoid misleading values.
                _terms = [f"data={loss_data.item():.4e}"]
                if self.lambda_recovery > 0.0:
                    _terms.append(f"recov={loss_recovery.item():.4e}")
                if self.lambda_strain > 0.0:
                    _terms.append(f"strain={loss_strain.item():.4e}")
                if self.lambda_stress > 0.0:
                    _terms.append(f"stress={loss_stress.item():.4e}")
                if self.lambda_stress_rate > 0.0:
                    _terms.append(f"srate={loss_stress_rate.item():.4e}")
                if self.lambda_obs > 0.0:
                    _terms.append(f"obs={loss_obs.item():.4e}")
                if self.lambda_obs_rate > 0.0:
                    _terms.append(f"obs_rate={loss_obs_rate.item():.4e}")
                if self.lambda_rf > 0.0:
                    _terms.append(f"rm={loss_rf.item():.4e}")
                if self.lambda_pde > 0.0:
                    _terms.append(f"pde={loss_pde.item():.4e}")
                print(f"[Epoch {epoch:5d}] total={loss_total.item():.4e} | "
                      + " ".join(_terms))
                if adaptive_loss_balance and len(adapt_log_vars) > 0:
                    w_line = " ".join(
                        f"{k}={torch.exp(-v.detach()).item():.2f}"
                        for k, v in adapt_log_vars.items()
                    )
                    print(f"  [AdaptiveW] {w_line}")
                if np.isfinite(wlf_monitor) or np.isfinite(ear_monitor):
                    wlf_str = f"{wlf_monitor:.4e}" if np.isfinite(wlf_monitor) else "nan"
                    ear_str = f"{ear_monitor:.4e}" if np.isfinite(ear_monitor) else "nan"
                    print(f"  [IDMonitor] wlf={wlf_str} ear={ear_str} best_wlf={best_wlf_score:.4e} best_ear={best_ear_score:.4e}")
                print(f"  [Phase] {phase}  physics_scale={physics_scale:.3f}")
                if self.lambda_rf > 0.0 and np.isfinite(rm_pred_mean):
                    print(f"  [RM] mean|pred|={rm_pred_mean:.4f} N mm  "
                          f"mean|FE|={rm_target_mean:.4f} N mm  n={n_rf_terms}")
                print(
                    f"  [Shift] C1={C1_now:.3f}(T:{self.mat.TRUE_C1})  "
                    f"C2={C2_now:.2f}(T:{self.mat.TRUE_C2})  "
                    f"Ea_R={EaR_now:.0f}(T:{self.mat.TRUE_Ea_R})"
                )
                _gm = self.mat.g_matrix.detach().cpu().tolist()
                _tg = self.mat.TRUE_G
                print(
                    f"  [Prony] g2={_gm[2]:.3f}(T:{_tg[2]})  g3={_gm[3]:.3f}(T:{_tg[3]})  "
                    f"g4={_gm[4]:.3f}(T:{_tg[4]})  g_inf={_gm[6]:.4f}(T:{_tg[6]})"
                )
                print(f"  [LR] net={lr_net:.2e} shift={lr_sft:.2e}  "
                      f"t={t_range} T={T_range} N_t={N_t}")

        # NOTE: the phase-best monitor snapshots are intentionally NOT restored.
        # The observation-window monitor (wlf/ear residual proxy) does not track
        # parameter accuracy, and restoring an earlier snapshot rolled the shift
        # parameters away from their converged (near-truth) end-of-training
        # values. We keep the trained-end parameters; the best monitor scores are
        # reported only for diagnostics.
        with torch.no_grad():
            _clamp_shift_params()
        if np.isfinite(best_wlf_score) or np.isfinite(best_ear_score):
            wlf_str = f"{best_wlf_score:.4e}" if np.isfinite(best_wlf_score) else "nan"
            ear_str = f"{best_ear_score:.4e}" if np.isfinite(best_ear_score) else "nan"
            print(f"  best monitors (diagnostic, not restored): wlf={wlf_str}, ear={ear_str}")

        print(f"\nTraining complete: {time.time() - start_time:.1f}s")

    # ------------------------------------------------------------------
    # Persistence helpers
    # ------------------------------------------------------------------
    def save_loss_history(self, path):
        rows = []
        for i, (total, comps) in enumerate(
                zip(self.loss_history, self.loss_components_history)):
            row = {'epoch': i, 'loss_total': total}
            row.update(comps)
            rows.append(row)
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"Loss history saved to {path}")

    def save_param_history(self, path):
        rows = [{'epoch': i, **h} for i, h in enumerate(self.param_history)]
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"Parameter history saved to {path}")


# ---------------------------------------------------------------------------


## Visualization Helpers\n\nDefine plotting utilities for training histories and saved benchmark diagnostics.\n

In [ ]:
# 5. Visualisation helpers
# ---------------------------------------------------------------------------

def plot_training_history(solver, save_dir, filename='ex5_pinn_inverse_training_history.png'):
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 12

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(len(solver.loss_history))

    ax = axes[0]
    ax.semilogy(epochs, solver.loss_history, 'k-', linewidth=1.5, label='Total')
    if solver.loss_components_history:
        for key, col, ls in [
            ('data', 'C0', '-'), ('strain', 'C1', '-'),
            ('stress', 'C8', '-'), ('recovery', 'C9', '-'),
            ('obs', 'C2', '-'), ('obs_rate', 'C3', '-.'),
            ('rf', 'C4', '-'), ('pde', 'C5', '--'),
            ('bc_left', 'C6', ':'), ('bc_right', 'C7', ':'),
        ]:
            # LSTM variants do not track every component key (e.g., stress/obs).
            if not any(key in h for h in solver.loss_components_history):
                continue
            vals = [h.get(key, 0.0) for h in solver.loss_components_history]
            ax.semilogy(epochs, vals, color=col, linestyle=ls,
                        linewidth=1, label=key, alpha=0.7)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Training Loss')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which='both')

    ax = axes[1]
    if solver.param_history:
        ep = range(len(solver.param_history))
        C1_true  = InverseShiftParams.TRUE_C1
        C2_true  = InverseShiftParams.TRUE_C2
        EaR_true = InverseShiftParams.TRUE_Ea_R

        ax2 = ax.twinx()
        l1, = ax.plot(ep,  [h['C1']   for h in solver.param_history],
                      'C0-', linewidth=1.5, label=f'C1 (true={C1_true})')
        l2, = ax.plot(ep,  [h['C2']   for h in solver.param_history],
                      'C1-', linewidth=1.5, label=f'C2 (true={C2_true})')
        ax.axhline(C1_true, color='C0', linestyle='--', alpha=0.5)
        ax.axhline(C2_true, color='C1', linestyle='--', alpha=0.5)
        l3, = ax2.semilogy(ep, [h['Ea_R'] for h in solver.param_history],
                           'C2-', linewidth=1.5, label=f'Ea/R (true={EaR_true})')
        ax2.axhline(EaR_true, color='C2', linestyle='--', alpha=0.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('C1, C2')
        ax2.set_ylabel('Ea/R [K]')
        ax.set_title('Shift Parameter Convergence')
        ax.legend(handles=[l1, l2, l3], fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = Path(save_dir) / filename
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Training history saved to {save_path}")
    plt.close()


# ---------------------------------------------------------------------------


## Training Configuration and Execution\n\nConfigure the bending benchmark, instantiate all components, run training/evaluation, and save outputs.\n

In [ ]:
# 6. Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(
        description="EX5 single-seed non-LSTM total-Lagrangian bending inverse."
    )
    parser.add_argument('--epochs', type=int, default=1200,
                        help='Training epochs.')
    parser.add_argument('--batch-size', type=int, default=96,
                        help='Spatial batch size per sampled time step.')
    parser.add_argument('--seq-length', type=int, default=24,
                        help='Temporal sequence length sampled each epoch.')
    parser.add_argument('--lr-network', type=float, default=2e-3,
                        help='Learning rate for field network.')
    parser.add_argument('--lr-params', type=float, default=2e-3,
                        help='Learning rate for shift parameters (default: 2e-3, '
                             'retained from the EX5 recovery pilot).')
    parser.add_argument('--adaptive-loss-balance', dest='adaptive_loss_balance', action='store_true', default=True,
                        help='Enable uncertainty-based adaptive balancing among data/obs/rf/pde losses.')
    parser.add_argument('--no-adaptive-loss-balance', dest='adaptive_loss_balance', action='store_false',
                        help='Disable adaptive loss balancing.')
    parser.add_argument('--adaptive-lr', dest='adaptive_lr', action='store_true', default=True,
                        help='Enable plateau-triggered adaptive learning-rate reduction.')
    parser.add_argument('--no-adaptive-lr', dest='adaptive_lr', action='store_false',
                        help='Disable adaptive learning-rate logic.')
    parser.add_argument('--field-block', type=int, default=2,
                        help='Field-only epochs per alternating cycle after warm-up.')
    parser.add_argument('--material-block', type=int, default=8,
                        help='Material-only epochs per alternating cycle after warm-up.')
    parser.add_argument('--stride', type=int, default=5,
                        help='Frame stride for loading FE results (used when --n-temporal<=0).')
    parser.add_argument('--n-temporal', type=int, default=100,
                        help='Sparse temporal subsampling: target total frames to load '
                             'across the cycle (~1296 available). 0 or negative → use --stride.')
    parser.add_argument('--n-spatial', type=int, default=1500,
                        help='Sparse spatial subsampling: number of nodes to keep '
                             '(end faces kept in full). 0 or negative → keep all 14454.')
    parser.add_argument('--seed', type=int, default=42,
                        help='Global random seed.')
    parser.add_argument('--lambda-rm', type=float, default=1.0,
                        help='Weight of the TL reaction-moment loss (default: 1).')
    parser.add_argument('--lambda-recovery', type=float, default=None,
                        help='Weight of the normalized EX5 free-recovery rotation '
                             'curve. Default: 80 in shift-only mode and 0 in the '
                             'legacy six-parameter audit mode.')
    parser.add_argument('--no-rm', dest='lambda_rm', action='store_const', const=0.0,
                        help='Disable RM supervision for the matched no-RM baseline.')
    parser.add_argument('--identify-prony', action='store_true', default=False,
                        help='Audit mode: jointly identify g2,g3,g4,g_inf. By default '
                             'the independent EX5 reference Prony spectrum is fixed and '
                             'only C1 and Ea/R are identified.')
    parser.add_argument('--output-dir', type=str, default=None,
                        help='Output directory (default: EX5-TL-INVERSE/seed_<seed>).')
    args = parser.parse_args(args=[])
    if args.lambda_recovery is None:
        args.lambda_recovery = 0.0 if args.identify_prony else 80.0
    set_global_seed(args.seed)
    n_spatial = args.n_spatial if args.n_spatial and args.n_spatial > 0 else None
    n_temporal = args.n_temporal if args.n_temporal and args.n_temporal > 0 else None

    print("=" * 70)
    print("EX5 SINGLE-SEED NON-LSTM TOTAL-LAGRANGIAN BENDING INVERSE")
    print("Bending shape-memory cycle (95x13x2 beam, end rotation UR=4.71 rad)")
    print("Configuration: " + (
        f"total-Lagrangian reaction-moment supervision at RP2 (lambda={args.lambda_rm:g})"
        if args.lambda_rm > 0.0 else "matched no-RM baseline"
    ))
    print("Inverse parameterization: " + (
        "six-parameter audit (C1, Ea/R, g2, g3, g4, g_inf; C2 fixed)"
        if args.identify_prony else
        "shift-only (C1 and Ea/R; C2 and independent EX5 Prony spectrum fixed)"
    ))
    print(f"Recovery-rotation supervision: lambda={args.lambda_recovery:g}")
    print("=" * 70)
    print(f"Seed: {args.seed}")

    script_dir = NOTEBOOK_DIR
    default_run_name = (f'six_param_audit_seed_{args.seed}' if args.identify_prony
                        else f'shift_only_seed_{args.seed}')
    output_dir = (Path(args.output_dir).expanduser().resolve()
                  if args.output_dir else script_dir / 'EX5-TL-INVERSE' / default_run_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {output_dir}")
    data_dir   = script_dir / 'EX-5-RESULTS'
    rf_file    = script_dir / 'ex-5-Bending-RM.csv'   # reaction moment (validation only)
    u_file     = script_dir / 'ex-5-Bending-UR.csv'   # end rotation (validation only)
    ft_file    = script_dir / 'frame-time.csv'

    print("\nLoading FE data...")
    fe_loader = FEDataLoader(data_dir, rf_file, u_file, ft_file, stride=args.stride,
                             n_spatial=n_spatial, spatial_seed=args.seed,
                             n_temporal=n_temporal)
    fe_loader.load_all_data()
    bounds = fe_loader.get_domain_bounds()

    print("\nDomain bounds:")
    for k, v in bounds.items():
        print(f"  {k}: {v:.2f}")

    # Characteristic displacement scale for output scaling (large-deformation bending).
    u_mag = np.linalg.norm(
        fe_loader.full_data[['U1', 'U2', 'U3']].values.astype(float), axis=1)
    u_scale = float(1.1 * np.max(u_mag))
    print(f"\nOutput displacement scale: {u_scale:.2f} mm (1.1 x max|u|)")

    print("\nInitialising identified parameters:")
    print(f"  C1   : init=8.0     true={InverseShiftParams.TRUE_C1}   (identified)")
    print(f"  C2   : FIXED=45.6   true={InverseShiftParams.TRUE_C2}   (held: non-identifiable)")
    print(f"  Ea/R : init=20000   true={InverseShiftParams.TRUE_Ea_R} (identified, via continuity)")
    mat_params = InverseShiftParams().to(device)
    if args.identify_prony:
        print(f"  Prony: g2,g3,g4,g_inf identified jointly (init≈0.350/0.250/0.080/0.020,")
        print(f"         true 0.306/0.358/0.034/0.002); g0,g1,g5 held fixed")
    else:
        mat_params.set_spectrum_to_true()
        mat_params.use_exact_reference_spectrum = True
        print("  Prony: FIXED to the independently prescribed EX5 reference spectrum")
        print("         [0.206, 0.093, 0.306, 0.358, 0.034, 0.001, 0.002]")
        print("  Branch stiffness: exact component-wise EX5 *User Material constants")

    model = SpatiotemporalPINN(hidden=(128, 128, 128, 128),
                               output_scale=u_scale).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nPINN parameters: {n_params:,}")

    # No FE strain/stress or analytic recovery target enters training. The inverse
    # gradient follows u_theta -> F -> E_mech -> q(a_T) -> S -> P -> equilibrium.
    solver = InversePINNSolver(
        model, mat_params, fe_loader, bounds,
        lambda_data=8.0,
        lambda_strain=0.0,
        lambda_recovery=args.lambda_recovery,
        lambda_stress=0.0,
        lambda_stress_rate=0.0,
        lambda_pde=1.0,
        lambda_bc_left=2.0,
        lambda_bc_right=8.0,
        lambda_traction=2.0,
        lambda_rf=args.lambda_rm,
        lambda_obs=0.0,
        lambda_obs_rate=0.0,
        finite_strain=True,
        identify_from_fe_strain=False,
        fix_c2=True,
        lambda_jacobian=2.0,
        full_history_grad=True,
        lambda_parameter_penalty=1.0,
        fix_prony=not args.identify_prony,
    )

    solver.train(
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr_network=args.lr_network,
        lr_params=args.lr_params,
        log_interval=100,
        seq_length=args.seq_length,
        adaptive_loss_balance=args.adaptive_loss_balance,
        adaptive_lr=args.adaptive_lr,
        field_block=args.field_block,
        material_block=args.material_block,
    )

    # Save results
    print("\nSaving results...")
    loss_csv  = output_dir / 'ex5_tl_inverse_loss_history.csv'
    param_csv = output_dir / 'ex5_tl_inverse_param_history.csv'
    solver.save_loss_history(loss_csv)
    solver.save_param_history(param_csv)

    # Visualise
    plot_training_history(solver, output_dir,
                          filename='ex5_tl_inverse_training_history.png')

    # Save checkpoint
    model_path = output_dir / 'ex5_tl_inverse_model.pth'
    torch.save({
        'model_state_dict':     model.state_dict(),
        'mat_params_state_dict': mat_params.state_dict(),
        'final_params': {
            'C1':   mat_params.C1.item(),
            'C2':   mat_params.C2.item(),
            'Ea_R': mat_params.Ea_R.item(),
        },
        'true_params': {
            'C1':   InverseShiftParams.TRUE_C1,
            'C2':   InverseShiftParams.TRUE_C2,
            'Ea_R': InverseShiftParams.TRUE_Ea_R,
        },
        'final_prony_spectrum': mat_params.g_matrix.detach().cpu().tolist(),
        'inverse_unknowns': (['C1', 'Ea_R', 'g2', 'g3', 'g4', 'g_inf']
                             if args.identify_prony else ['C1', 'Ea_R']),
        'use_exact_reference_spectrum': bool(mat_params.use_exact_reference_spectrum),
        'run_config': vars(args),
        'formulation': 'total_lagrangian_network_kinematics',
    }, model_path)
    print(f"Model saved to {model_path}")

    # Final summary
    print("\n" + "=" * 70)
    print("FINAL IDENTIFIED PARAMETERS (EX5 PINN, TL RM, " +
          ("shift + Prony spectrum)" if args.identify_prony else "shift-only)"))
    print("=" * 70)
    print(f"  C1   = {mat_params.C1.item():.4f}   (true: {InverseShiftParams.TRUE_C1})")
    print(f"  C2   = {mat_params.C2.item():.4f} K (true: {InverseShiftParams.TRUE_C2} K)")
    print(f"  Ea/R = {mat_params.Ea_R.item():.2f} K (true: {InverseShiftParams.TRUE_Ea_R} K)")
    c1_err  = abs(mat_params.C1.item()  - InverseShiftParams.TRUE_C1)  / InverseShiftParams.TRUE_C1 * 100
    c2_err  = abs(mat_params.C2.item()  - InverseShiftParams.TRUE_C2)  / InverseShiftParams.TRUE_C2 * 100
    ear_err = abs(mat_params.Ea_R.item()- InverseShiftParams.TRUE_Ea_R)/ InverseShiftParams.TRUE_Ea_R* 100
    print(f"\n  Relative errors: C1={c1_err:.1f}%  C2={c2_err:.1f}%  Ea/R={ear_err:.1f}%")

    # Identified Prony spectrum {g2,g3,g4,g_inf}
    _gm = mat_params.g_matrix.detach().cpu().tolist()
    _tg = InverseShiftParams.TRUE_G
    print("\n  Prony spectrum (" +
          ("identified: g2,g3,g4,g_inf; fixed: g0,g1,g5" if args.identify_prony
           else "fixed independently at the EX5 reference") + "):")
    for _k, _name in ((2, 'g2'), (3, 'g3'), (4, 'g4'), (6, 'g_inf')):
        _e = abs(_gm[_k] - _tg[_k]) / max(_tg[_k], 1e-9) * 100
        print(f"    {_name:5s} = {_gm[_k]:.4f}   (true: {_tg[_k]:.4f})   err={_e:.1f}%")




## Notebook Entry Point\n\nRun the selected benchmark when the notebook is executed as a script.\n

In [ ]:
if __name__ == "__main__":
    _script_dir = NOTEBOOK_DIR
    _log_path = _script_dir / 'ex5_tl_inverse_training_log.txt'
    _tee = _Tee(sys.stdout, _log_path)
    sys.stdout = _tee
    try:
        main()
    finally:
        sys.stdout = _tee._orig
        _tee.close()
        _tee._orig.write(f"\nLog saved to: {_log_path}\n")
